# 2. Training and testing ArChIPelago

In [ ]:
import os
import re
import sys
import argparse
import subprocess
import matplotlib
import time
import random
import string
import shlex
import shutil
import glob
import pickle
import csv
import operator
import joblib
import pandas as pd
import numpy as np
from itertools import groupby
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import defaultdict
from sklearn.ensemble import BaggingClassifier
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO
from sklearn.feature_selection import RFE
from sklearn import preprocessing
import pybedtools as pbt
import pyBigWig as pbw
from datetime import date
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from itertools import chain
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from sklearn.ensemble import AdaBoostClassifier
from adjustText import adjust_text
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline
from sklearn.preprocessing import StandardScaler

In [ ]:
#os.chdir("/home/pavelkrav/")

root = os.getcwd()
basicdir = os.path.abspath('Release/TF-ML/')
outputdir = os.path.abspath('Release/TF-ML/outputdir')
train_dir = os.path.abspath('Release/TF-ML/train/') 
test_dir = os.path.abspath('Release/TF-ML/test/') 

if not os.path.exists(basicdir):
    os.makedirs(basicdir)
if not os.path.exists(outputdir):
    os.makedirs(outputdir)
if not os.path.exists(train_dir):
    os.makedirs(train_dir)
if not os.path.exists(test_dir):
    os.makedirs(test_dir)


pwmdir_mono = os.path.abspath('./hocomoco11/models/pwm/mono/all')
pwmdir_di = os.path.abspath('./hocomoco11/models/pwm/di/all')


files_pwm_HUMAN_mono = []
files_pwm_MOUSE_mono = []

for d in os.listdir(pwmdir_mono):
    d = pwmdir_mono + "/" + d 
    if os.path.isdir(d):
        os.chdir(d)
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm'])
    os.chdir(pwmdir_mono)

print(len(files_pwm_HUMAN_mono))
print(len(files_pwm_MOUSE_mono))


files_pwm_HUMAN_di = []
files_pwm_MOUSE_di = []

for d in os.listdir(pwmdir_di):
    d = pwmdir_di + "/" + d
    if os.path.isdir(d): 
        os.chdir(d) 
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm'])
    os.chdir(pwmdir_di) 

print(len(files_pwm_HUMAN_di))
print(len(files_pwm_MOUSE_di))


def remove_duplicates(l):
    return list(set(l))

print("We have {N} TF before filtration of mono".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_mono + files_pwm_HUMAN_mono]))))
print("We have {N} TF before filtration of di".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_di + files_pwm_HUMAN_di]))))

In [ ]:
size_of_set = 10000
outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_control.fasta"

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

In [ ]:
headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_control.csv"

In [ ]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [ ]:
trimmed = False
size = 20
begin_trim = size
end_trim = size
if trimmed == True:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
else:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

In [ ]:
with open(headerStr_names_outfile_HUMAN_train, "r") as output:
    headerStr_names_HUMAN_train = output.read().split('\n')
    print(headerStr_names_HUMAN_train[:10])

with open(headerStr_names_outfile_MOUSE_train, "r") as output:
    headerStr_names_MOUSE_train = output.read().split('\n')
    print(headerStr_names_MOUSE_train[:10])

with open(headerStr_names_outfile_HUMAN_control, "r") as output:
    headerStr_names_HUMAN_control = output.read().split('\n')
    print(headerStr_names_HUMAN_control[:10])

with open(headerStr_names_outfile_MOUSE_control, "r") as output:
    headerStr_names_MOUSE_control = output.read().split('\n')
    print(headerStr_names_MOUSE_control[:10])    

In [ ]:
family_annotation = pd.read_csv("/hocomoco11/core_annotation_HUMAN_mono.csv", sep='\t')

def find_TF_family_dict(family_annotation):
    TF_family_dict = defaultdict(None)
    for i, j in zip(family_annotation["Model"], family_annotation["TF family"]):
        if type(j) == str:
            TF_family_dict[i.split('_')[0]] = j.split('{')[1][:3]
    return TF_family_dict

TF_family_dict = find_TF_family_dict(family_annotation)

In [ ]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [ ]:
selected_HUMAN = ['ANDR_HUMAN',
 'AP2A_HUMAN',
 'CEBPB_HUMAN',
 'COE1_HUMAN',
 'CTCF_HUMAN',
 'E2F4_HUMAN',
 'ERG_HUMAN',
 'ESR1_HUMAN',
 'FLI1_HUMAN',
 'GATA1_HUMAN',
 'GATA2_HUMAN',
 'GATA3_HUMAN',
 'GCR_HUMAN',
 'HNF4A_HUMAN',
 'IRF1_HUMAN',
 'IRF4_HUMAN',
 'JUND_HUMAN',
 'MAFK_HUMAN',
 'MAX_HUMAN',
 'MYC_HUMAN',
 'P53_HUMAN',
 'PPARG_HUMAN',
 'PRGR_HUMAN',
 'REST_HUMAN',
 'RUNX1_HUMAN',
 'RXRA_HUMAN',
 'SOX2_HUMAN',
 'SPI1_HUMAN',
 'SRF_HUMAN',
 'STA5A_HUMAN',
 'STAT1_HUMAN',
 'STAT3_HUMAN',
 'TAL1_HUMAN',
 'TF65_HUMAN',
 'TFE2_HUMAN',
 'USF2_HUMAN']

selected_MOUSE = ['ANDR_MOUSE',
 'AP2A_MOUSE',
 'BATF_MOUSE',
 'CEBPB_MOUSE',
 'COE1_MOUSE',
 'CTCF_MOUSE',
 'E2F4_MOUSE',
 'ERG_MOUSE',
 'ESR1_MOUSE',
 'FLI1_MOUSE',
 'GATA1_MOUSE',
 'GATA2_MOUSE',
 'GATA3_MOUSE',
 'GCR_MOUSE',
 'HNF4A_MOUSE',
 'IRF1_MOUSE',
 'IRF4_MOUSE',
 'JUND_MOUSE',
 'MAFK_MOUSE',
 'MAX_MOUSE',
 'MYC_MOUSE',
 'MYOD1_MOUSE',
 'P53_MOUSE',
 'PPARG_MOUSE',
 'PRGR_MOUSE',
 'REST_MOUSE',
 'RUNX1_MOUSE',
 'RXRA_MOUSE',
 'SOX2_MOUSE',
 'SPI1_MOUSE',
 'SRF_MOUSE',
 'STA5A_MOUSE',
 'STAT1_MOUSE',
 'STAT3_MOUSE',
 'TAL1_MOUSE',
 'TF65_MOUSE',
 'TFE2_MOUSE',
 'USF2_MOUSE']

In [ ]:
def fasta_iter(f):
    faiter = (x[1] for x in groupby(f, lambda line: line[0] == ">"))

    for header in faiter:
        headerStr = header.__next__()[1:].strip()
        seq = "".join(s.strip() for s in faiter.__next__())
        yield (headerStr, seq)

        


root = "/home/pavelkrav"

os.chdir(root)
    
def filter_chroms(hg_index):
    reg = re.compile("NC_.*chromosome\ ([12]?[0-9XY])")
    for seq in hg_index.values():
        m = reg.match(seq.description)
        if m:
            name = f"chr{m.group(1)}"
            seq.id = name
            seq.description = ''
            if seq.description.startswith('NC'):
                print (seq.description)
            yield seq
            
def convert_genome_file(genomepath, convertedpath):
    human_genome = SeqIO.index(filename=genomepath,
                           format="fasta")
    with open(convertedpath, "w") as fd:
        SeqIO.write(filter_chroms(hg_index=human_genome),
                    fd,
                    'fasta')    

def convert_region_file(region_file, converted_file):
    data = pd.read_table(region_file, index_col=0)
    data['chrom_col'] = data['chrom_col'].apply(lambda x : f"chr{x}") 
    data['rel'] = 0
    data.to_csv(converted_file, sep="\t", header=False, index=None)
    

class SlimError(Exception):
    pass
    
class WrongJavaPath(SlimError):
    def __init__(self, msg, javapath, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.javapath=javapath
        self.msg = msg
        self.orig_exc = orig_exc

class WrongSlimPath(SlimError):
    def __init__(self, msg, slimpath, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.slimpath=slimpath
        self.msg = msg
        self.orig_exc = orig_exc

class EntityExists(SlimError):
    def __init__(self, msg, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.orig_exc = orig_exc
        
class EntityNotExists(SlimError):
    def __init__(self, msg, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.orig_exc = orig_exc

class ExtractRuntimeError(SlimError):
    def __init__(self, msg, stderr="", orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.stderr = stderr
        self.orig_exc = orig_exc
        

class InvalidPeaksFile(SlimError):
    def __init__(self, msg, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.orig_exc = orig_exc
        
class InvalidParameter(SlimError):
    def __init__(self, msg, orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.orig_exc = orig_exc
        
class NotExtractedDatasetUse(SlimError):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg

def validate_java(javapath):
    e = None
    msg = ''
    try:
        p = subprocess.Popen(args=[javapath, '-version'], stdout=subprocess.PIPE,
                stderr=subprocess.PIPE)
        print(p)
    except FileNotFoundError as exc:
        e = exc
        msg = 'no such file'
    except PermissionError as exc:
        e = exc
        msg = "permission error"
    except Exception as exc:
        e = exc
        msg = "unknown exception"
    if e is not None:
        raise WrongJavaPath("Wrong java path: {}".format(msg), javapath=javapath,
                           orig_exc=e)

    returncode = p.wait()
    if returncode != 0:
        raise WrongJavaPath("Wrong java path", javapath=javapath)

    output = p.stderr.read().decode().lower()
    if not 'openjdk' in output:
        if 'java' in output:
            raise WrongJavaPath('''Jan compiled his package using OpenJDK8. 
It won't work with usual Java.
Check how to install OpenJDK on your computer''',
                               javapath=javapath)
        else:
            raise WrongJavaPath("Wrong java path", javapath=javapath)
                    
def check_int_col(dt, ind, name):
    exc = None
    try:
        dt.iloc[:, ind].apply(lambda x : int(x))
    except Exception as e:
        exc = e
    if exc is not None:
        raise InvalidPeaksFile(f"{name} columns must contain only integers")
        
def check_float_col(dt, ind, name):
    exc = None
    try:
        dt.iloc[:, ind].apply(lambda x : float(x))
    except Exception as e:
        exc = e
    if exc is not None:
        raise InvalidPeaksFile(f"{name} columns must contain only numbers")

def check_re_col(dt, ind, name, regex):
    exc = None
    v = sum(dt.iloc[:,ind].apply(lambda x : regex.match(str(x)) is None))
    if v > 0:
        raise InvalidPeaksFile(f"{name} columns must contain only chromosomes in format {regex.pattern}")


CHR_COL_REGEX = re.compile("chr(([0-9]+)|[XY])")

def validate_peaks_file(peaks_file,
                        chr_col=1, # 1-based indexing,
                        peak_col=2, # peak center
                        sig_col=3, # peak signal,
                        offset_col=4 # peak_col + offset_col = absolute position of peak in genome
                       ):
    data = pd.read_table(peaks_file)
    if data.shape[1] < chr_col:
        raise InvalidPeaksFile(f"chr col position in file {peaks_file} is out of range: {chr_col}")
    if data.shape[1] < peak_col:
        raise InvalidPeaksFile(f"peak col position in file {peaks_file} is out of range: {peak_col}")
    if data.shape[1] < sig_col:
        raise InvalidPeaksFile(f"sig col position in file {peaks_file} is out of range: {sig_col}")
    if data.shape[1] < offset_col:
        raise InvalidPeaksFile(f"offset col position in file {peaks_file} is out of range: {offset_col}")
    
    check_re_col(data, chr_col - 1, 'chr', CHR_COL_REGEX)
    check_int_col(data, peak_col - 1, 'peak')
    check_int_col(data, offset_col - 1, 'offset')
    check_float_col(data, sig_col - 1, 'signal')
        
    



def randomStringDigits(stringLength=10):
    import string
    import random
    """Generate a random string of letters and digits """
    lettersAndDigits = string.ascii_letters + string.digits
    return ''.join(random.choice(lettersAndDigits) for i in range(stringLength))    
           
    
class SlimDataset:
    def __init__(self, genome_file, 
                 peaks_file,
                 seq_width=1000, # width of window around peak
                 dataset_dir = "SLIM_WORKING_DIR/DATASETS",
                 log_dir="SLIM_WORKING_DIR/DATASETS/logs/",
                 exec_path="/home/pavelkrav/Slim/TrainAndApplySlim.jar",
                 javapath="/home/pavelkrav/Slim/jdk8u232-b09/bin/java",
                 outfile_name=None, # generate random name
                 log_level=2, # 1 to output only python messages, 0 to no log, 2 - output Java messages also
                 chr_col=1, # 1-based indexing,
                 peak_col=2, # peak center
                 sig_col=3, # peak signal,
                 offset_col=4 # peak_col + offset_col = absolute position of peak in genome
                ):
        
        self.genome_file = genome_file
        self.peaks_file = peaks_file
        self.seq_width = seq_width
        self.dataset_dir = dataset_dir
        self.log_dir = log_dir
        self.exec_path = exec_path
        self.javapath = javapath
        self.__outfile_name = outfile_name or randomStringDigits()
        self.log_level=log_level
        
        self.chr_col = chr_col
        self.peak_col = peak_col
        self.sig_col = sig_col
        self.offset_col = offset_col
        self._dataset = None
        self._no_messages = None
        
        self.validate()
        
        if not os.path.exists(self.dataset_dir):
            os.makedirs(self.dataset_dir)
        
        if not os.path.exists(self.log_dir):
            os.makedirs(self.log_dir)

    def validate(self): 
        if not os.path.exists(self.genome_file):
            raise EntityNotExists(f"Genome file: {self.genome_file} doesn't exist")
        if not os.path.exists(self.peaks_file):
            raise EntityNotExists(f"Peaks file: {self.peaks_file} doesn't exist")
        if not os.path.exists(self.exec_path):
            raise EntityNotExists(f"Exec file: {self.exec_path} doesn't exist")
        
        if  self.seq_width <= 0 or self.seq_width > 10000:
            raise InvalidParameter(f"seq_width is not in [1,10000]: {self.seq_width}")

        validate_java(self.javapath)
        validate_peaks_file(self.peaks_file, 
                            chr_col=self.chr_col,
                            peak_col=self.peak_col,
                            sig_col=self.sig_col,
                            offset_col=self.offset_col)
        
    @property
    def dataset_path(self):
        if self._dataset is None:
            self.run_extract()
        return self._dataset
        
    @property
    def no_messages(self): # messages about missing chromosomes or peaks
        if self._no_messages is None:
            raise NotExtractedDatasetUse("Dataset isn't extracted, so no no-messages exist")
        return self._no_messages
    
    def run_extract(self):
        
        outfile_tempdir = self.__outfile_name + "tempdir"
        
        e = None
        try:
            os.mkdir(outfile_tempdir)
        except FileExistsError as exc:
            e = exc
        if e is not None:
            raise EntityExists(f"Entity {outfile_tempdir} already exists", e)
            
        command = f"{self.javapath} -jar {self.exec_path} extract "\
                  f"g={self.genome_file} p={self.peaks_file} "\
                  f"w={self.seq_width} outdir={outfile_tempdir} "\
                  f"c={self.chr_col} cc={self.peak_col} "\
                  f"sc={self.sig_col} s={self.offset_col}"
        e = None
        try:
            p = subprocess.Popen(shlex.split(command), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)

        except Exception as exc:
            e = exc
        if e is not None:
            raise ExtractRuntimeError("Error running extract", stderr="", orig_exc=e)
        
        if self.log_level >= 1:
            print("Extracting dataset", file=sys.stderr)
        
        if self.log_level >= 2:
            lines =[]
            returncode = p.poll()
            while returncode is None:
                while True:
                    line = p.stdout.readline().decode()
                    lines.append(line)
                    if not line:
                        break
                    print(line, file=sys.stderr, end="")
                returncode = p.poll()
        else:
            returncode = p.wait()
            
            lines = p.stdout.read().decode().splitlines()
        
        

        if returncode:
            stderr = p.stdout.read().decode()
            raise ExtractRuntimeError(f"Error running extract: {stderr}", stderr=stderr)
        
        self._no_messages = [m.strip() for m in lines if m.startswith("No ")]
        
        temp_out = os.path.join(outfile_tempdir, "Extracted_sequences.fasta")
        real_out = os.path.join(self.dataset_dir, self.__outfile_name) + ".fasta"
        shutil.move(temp_out, real_out)
        self._dataset = real_out
        
        temp_logfile = os.path.join(outfile_tempdir, "protocol_extract.txt")
        real_logfile = os.path.join(self.log_dir, self.__outfile_name) + ".log"
        shutil.move(temp_logfile, real_logfile)
        
        shutil.rmtree(outfile_tempdir)       
        
        if self.log_level >= 1:
            print("Done", file=sys.stderr)


class SlimDimontRuntimeError(Exception):
    def __init__(self, msg, stderr="", orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.stderr = stderr
        self.orig_exc = orig_exc

class SlimScanRuntimeError(Exception):
    def __init__(self, msg, stderr="", orig_exc = None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.stderr = stderr
        self.orig_exc = orig_exc
        
class SlimDimontUnfitted(Exception):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        
class SlimDimontTrainedOnce(Exception):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg

class SlimDimontWrongTrainFile(Exception):
    def __init__(self, msg, seq_ind=None, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        self.seq_ind = None

class SlimDimontWrongBackgroundFile(Exception):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        
class SlimScanWrongModelId(Exception):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg
        
class SlimScanWrongModelPath(Exception):
    def __init__(self, msg, *args, **kwargs):
        super().__init__(msg, *args, **kwargs)
        self.msg = msg

def is_fasta_file(fasta_file):
    fastagen = SeqIO.parse(fasta_file, format='fasta')
    try:
        next(fastagen)
    except (ValueError, StopIteration) as exc:
        return False
    return True
    
        
def validate_train_file(input_file, peak_field, value_field):
    if not os.path.exists(input_file):
        raise SlimDimontWrongTrainFile("Train file doesn't exist")
    
    if not is_fasta_file(input_file):
        raise SlimDimontWrongTrainFile("Train file is not in .fasta-format")
    
    fastagen = SeqIO.parse(input_file, format='fasta')
        
    peak_field = peak_field.strip()
    value_field = value_field.strip()

    for ind, seq in enumerate(fastagen):
        keys = [x.split(":")[0].strip() for x in seq.description.split(";")]
        pfc = keys.count(peak_field)
        if pfc < 1:
            raise SlimDimontWrongTrainFile(
               f"At least one sequence (ind={ind}) in train file doesn't contain peak field: {peak_field}",
            seq_ind=ind)
        elif pfc > 1:
            raise SlimDimontWrongTrainFile(
               f"At least one sequence (ind={ind}) in train file contains duplicated peak field: {peak_field}",
            seq_ind=ind)
        vfc = keys.count(value_field)
        if vfc < 1:
            raise SlimDimontWrongTrainFile(
               f"At least one sequence (ind={ind}) in train file doesn't contain value field: {value_field}",
            seq_ind=ind)
        elif vfc > 1:
            raise SlimDimontWrongTrainFile(
               f"At least one sequence (ind={ind}) in train file contains duplicated value field: {value_field}",
            seq_ind=ind)

def validate_background_file(background_file):
    if not os.path.exists(background_file):
        raise SlimDimontWrongBackgroundFile("Background file doesn't exist")
    
    if not is_fasta_file(background_file):
        raise SlimDimontWrongBackgroundFile("Background file is not in fasta format")
        
LOG_LINE_REGEX = re.compile("([^-]*)\s- .*\s=\s([^=]+)$")
def parse_slim_log(logfilepath):
    with open(logfilepath, "r") as logfile:
        config = {}
        for line in logfile:
            if line.startswith("-----------------------------------------"):
                break
            m = LOG_LINE_REGEX.match(line)
            if m:
                key = m.group(1)
                value = m.group(2).strip()
                config[key] = value
    return config

def load_slim_prediction(slim_prediction_file, pref=""):
    table = pd.read_table(slim_prediction_file,
                          index_col=0, 
                          header=None)
    table.index.name = f'seq_ind'
    table.columns = [f'predicted_center{pref}',
                     f'strand{pref}', 
                     f'maxscore{pref}', 
                     f'sumscore{pref}', 
                     f'motif{pref}', 
                     f'metainfo']
    
    return table



class SlimModel:
    @staticmethod
    def get_model_id(path):
        try:
            no_ext = os.path.splitext(path)[0]
            str_id = no_ext.rsplit("_", 1)[-1]
        except IndexError:
            raise SlimScanWrongModelPath("Wrong path for the model")
        else:
            try:
                 m_id = int(str_id)
            except ValueError:
                raise SlimScanWrongModelPath("Wrong path for the model")
        return m_id
    
    def __init__(self, 
                 markov_order,
                 background_markov_order=-1, # Markov order of background model
                 weighting_factor=0.2, 
                 preopt_runs=20, # The number of pre-optimization runs
                 out_dir_name=None, # generate random name
                 model_dir = "SLIM_WORKING_DIR/MODELS",
                 log_dir="SLIM_WORKING_DIR/MODELS/logs/",
                 threads=1,
                 exec_path="/home/pavelkrav/Slim/TrainAndApplySlim.jar",
                 javapath='/home/pavelkrav/Slim/jdk8u232-b09/bin/java',
                 train_once=False,
                 log_level=2, # 1 to output only python messages, 0 to no log, 2 - output Java messages also
                ):
        
        self.markov_order = markov_order
        self.background_markov_order = background_markov_order
        self.out_dir_name = out_dir_name or randomStringDigits()
        self.model_dir = model_dir
        self.log_dir = log_dir
        self.threads = threads
        self.exec_path = exec_path
        self.javapath = javapath
        self.log_level = log_level
        self.train_once = train_once
        self.preopt_runs = preopt_runs
        self.weighting_factor = weighting_factor
        
        self.validate()
        
        out_dir = os.path.join(self.model_dir, self.out_dir_name)
        if os.path.exists(out_dir):
            print("Reading pretrained model", file=sys.stderr)
            
            models = glob.glob(os.path.join(out_dir, "Motif_*/*.xml"))
            ids = [SlimModel.get_model_id(m) for m in models]
            self._models = {i : m for i, m in zip(ids, models)}
        else:
            self._models = None
            
            
        if not os.path.exists(self.model_dir):
            os.makedirs(self.model_dir)
        
        if not os.path.exists(self.log_dir):
            os.makedirs(self.log_dir)
    
    def validate(self):
        if not os.path.exists(self.exec_path):
            raise EntityNotExists(f"Exec file: {self.exec_path} doesn't exist")

        if  self.markov_order < -2147483648 or self.markov_order > 3:
            raise InvalidParameter(f"markov order is not in [-2147483648, 3]: {self.markov_order}")
            
        if self.background_markov_order < -1 or self.background_markov_order > 5:
            raise InvalidParameter(f"background markov order is not in [-1, 5]: {self.background_markov_order}")
            
        if self.threads < 0:
            raise InvalidParameter(f"The number of threads must be positive: {self.threads}")

        if self.preopt_runs < 1 or self.preopt_runs > 100:
            raise InvalidParameter(f"The number of preoptimization runs must be in [1,100]: {self.preopt_runs}")
            
        if isinstance(self.weighting_factor, float):
            if self.weighting_factor <= 0 or self.weighting_factor >= 1:
                raise InvalidParameter(f"The weighting factor must be  in (0, 1): {self.weighting_factor}")
        elif isinstance(self.weighting_factor, str):
            val = self.weighting_factor.split("sd")[0]
            try:
                val = float(val)
            except:
                raise InvalidParameter(f"The weighting factor must be number of in {{n}}sd format (n can be float): {self.weighting_factor}")
        else:
            raise InvalidParameter(f"The weighting factor must be number or in {{n}}sd format (n can be float): {self.weighting_factor}")
            
        validate_java(self.javapath)
        
        
    def fit(self, input_file, background_file=None, peak_field="peak", value_field="signal"):
        
        if self.log_level >= 1:
            print("Checking files", file=sys.stderr)
        validate_train_file(input_file, peak_field=peak_field, value_field=value_field)
        if background_file is not None:
            validate_background_file(background_file)
        
        out_dir = os.path.join(self.model_dir, self.out_dir_name)
        if os.path.exists(out_dir):
            if self.train_once:
                raise SlimDimontTrainedOnce("Model is alredy trained and train_once is set to True")
            else:
                shutil.rmtree(out_dir) # clean up results of previous fit

        os.makedirs(out_dir)
        
        
        if background_file is not None:
            command = f"{self.javapath} -jar {self.exec_path} "\
                      f"slimdimont i={input_file} b={background_file} "\
                      f"p={peak_field} v={value_field} "\
                      f"m={self.markov_order} moobm={self.background_markov_order} "\
                      f"w={self.weighting_factor} Starts={self.preopt_runs} "\
                      f"outdir={out_dir} threads={self.threads} "
        else:
            command = f"{self.javapath} -jar {self.exec_path} "\
                      f"slimdimont i={input_file} "\
                      f"p={peak_field} v={value_field} "\
                      f"m={self.markov_order} moobm={self.background_markov_order} "\
                      f"w={self.weighting_factor} Starts={self.preopt_runs} "\
                      f"outdir={out_dir} threads={self.threads} "
        
        
        if self.log_level >= 1:
            print("Running slimdimont", file=sys.stderr)     
        e = None
        try:
            p = subprocess.Popen(shlex.split(command), 
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT)

        except Exception as exc:
            e = exc
        if e is not None:
            raise SlimDimontRuntimeError("Error running slimdimont", stderr="", orig_exc=e)
        
        
        if self.log_level >= 2:
            returncode = p.poll()
            while returncode is None:
                while True:
                    line = p.stdout.readline().decode()
                    if not line:
                        break
                    print(line, file=sys.stderr, end="")
                returncode = p.poll()
        else:
            returncode = p.wait()
            
        if returncode:
            stderr = p.stdout.read().decode()
            raise SlimDimontRuntimeError(f"Error running slimdimont: {stderr}", stderr=stderr)
        elif self.log_level >= 1:
            print("Done", file=sys.stderr)   
        
            models = glob.glob(os.path.join(out_dir, "Motif_*/*.xml"))
            ids = [SlimModel.get_model_id(m) for m in models]
            self._models = {i : m for i, m in zip(ids, models)}
        
        templog = os.path.join(out_dir, 'protocol_slimdimont.txt')
        modellogfile = os.path.join(self.log_dir, self.out_dir_name) + ".log" 
        shutil.move(templog, modellogfile)
        

        
    @property
    def models(self):
        if self._models is None:
            raise SlimDimontUnfitted("Model isn't fitted")
        return self._models
    
    def process_prediction(self, prediction_paths):
        df = load_slim_prediction(prediction_paths[0], 1)
        for ind, path in enumerate(prediction_paths[1:], 2):
            tab = load_slim_prediction(path, ind)
            tab.drop(labels=['metainfo'], axis=1, inplace=True)
            df = pd.merge(df, tab, 
                          how='outer',
                          validate="1:1",
                          left_index=True, 
                          right_index=True)
        return df
        
    
    @property
    def models_ids(self):
        return list(self._models.keys())
    
    def predict(self, seq_file, 
                models_ids='all', # use all models
                predictions_dir=None # generate random name
               ):  
        if predictions_dir is None:
            predictions_dir = randomStringDigits() + "_predictions"
            predictions_dir = os.path.join(self.model_dir, 
                                           self.out_dir_name,
                                           predictions_dir)
            
        os.makedirs(predictions_dir)
        
        if models_ids == "all":
            models_ids = self.models_ids
        else:
            for i in models_ids:
                if i not in self._models:
                    raise SlimScanWrongModelId(f"Wrong model id: {i}. Check available model ids using SlimModel.models_ids")
        
        predictions_files = []
        for i in models_ids:
            predictions_files.append(self._predict_single_model(i, seq_file, predictions_dir))
    
        return self.process_prediction(predictions_files)
        
    
    def _predict_single_model(self, mod_id, seq_file, predictions_dir):
        if self.log_level >= 1:
            print(f"Predicting using model {mod_id}", file=sys.stderr)
            
        out_dir = os.path.join(predictions_dir, randomStringDigits())
        os.mkdir(out_dir)
        
        command = f"{self.javapath} -jar {self.exec_path} scan "\
                  f"i={seq_file} m={self._models[mod_id]} "\
                  f"outdir={out_dir}"

        e = None
        try:
            p = subprocess.Popen(shlex.split(command),
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT)
        except Exception as exc:
            e = exc
        if e is not None:
            raise SlimScanRuntimeError("Error running scan", stderr="", orig_exc=e)
            
        if self.log_level >= 2:
            returncode = p.poll()
            while returncode is None:
                while True:
                    line = p.stdout.readline().decode()
                    if not line:
                        break
                    print(line, file=sys.stderr, end="")
                returncode = p.poll()
        else:
            returncode = p.wait()
            
        if returncode:
            stderr = p.stdout.read().decode()
            raise SlimScanRuntimeError(f"Error running scan: {stderr}", stderr=stderr)
        
        tempprediction_file = os.path.join(out_dir, 'predictions.txt')
        realprediction_file = os.path.join(predictions_dir,
                                          'model{}_predictions'.format(mod_id)) + '.txt'
        
        shutil.move(tempprediction_file, realprediction_file)
        shutil.rmtree(out_dir)
        return realprediction_file
        
        
    @classmethod
    def load(cls, model_name,
             model_dir="SLIM_WORKING_DIR/MODELS",
             log_dir="SLIM_WORKING_DIR/MODELS/logs/",
             exec_path="/home/pavelkrav/Slim/TrainAndApplySlim.jar",
             javapath='/home/pavelkrav/Slim/jdk8u232-b09/bin/java',
             no_retrain=True,
             log_level=2
             ):
        logfile = os.path.join(log_dir, model_name) + '.log'
        config = parse_slim_log(logfile)
        
        markov_order = int(config['m'])
        background_markov_order=int(config['moobm'])
        preopt_runs=int(config['Starts'])
        threads = int(config['threads'])
        if 'sd' in config['w']:
            weighting_factor = config['w']
        else:
            weighting_factor = float(config['w'])
        
        return cls(markov_order,
                 background_markov_order=background_markov_order,
                 weighting_factor=weighting_factor , 
                 preopt_runs=preopt_runs, 
                 out_dir_name=model_name,
                 model_dir = model_dir,
                 log_dir = log_dir,
                 threads=threads,
                 exec_path=exec_path,
                 javapath=javapath,
                 train_once=no_retrain,
                 log_level=log_level)



In [ ]:
from abc import ABCMeta, abstractmethod
from dataclasses import dataclass, field

from typing import List
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

@dataclass
class Scorer(metaclass=ABCMeta):
    name: str
    @abstractmethod
    def score(self, *args, **kwargs) -> float:
        pass

@dataclass
class ConstantScorer(Scorer):
    const: float
    def score(self, *args, **kwargs) -> float:
        return self.const

class BinaryScorer(Scorer):
    @abstractmethod
    def score(self, y_score: List[float], y_real: List[int]) -> float:
        raise NotImplementedError

class SklearnScorer(BinaryScorer):
    pass

class SklearnROCAUC(SklearnScorer):
    def score(self, y_score: List[float], y_real: List[int]) -> float:
        y_score_arr = np.array(y_score)
        y_real_arr = np.array(y_real)
        return float(roc_auc_score(y_true=y_real_arr, y_score=y_score_arr))
    
class SklearnPRAUC(SklearnScorer):
    def score(self, y_score: List[float], y_real: List[int]) -> float:
        y_score_arr = np.array(y_score)
        y_real_arr = np.array(y_real)
        return float(average_precision_score(y_true=y_real_arr, y_score=y_score_arr))

class PRROCScorer(BinaryScorer):
    pass

def import_PRROC():
    '''
    import PRROC package (https://cran.r-project.org/web/packages/PRROC/index.html)
    '''
    from rpy2.robjects.packages import importr, isinstalled
    if not isinstalled("PRROC"):
        utils = importr("utils")
        utils.chooseCRANmirror(ind=1)
        utils.install_packages("PRROC", quiet = True, verbose=False)
    pkg = importr("PRROC")
    return pkg

@dataclass
class PRROC_PRAUC(PRROCScorer):
    type: str

    def score(self, y_score: List[float], y_real: List[int]) -> float:
        from rpy2.rinterface_lib import openrlib
        with openrlib.rlock:
            pkg = import_PRROC()
            from rpy2.robjects.vectors import FloatVector
            labels = FloatVector([x for x in y_real])
            scores = FloatVector(y_score)
            if self.type == "integral":
                auroc = pkg.pr_curve(scores, weights_class0=labels, dg_compute=False)
                auroc = auroc[1][0]
            elif self.type == "davisgoadrich":
                auroc = pkg.pr_curve(scores, weights_class0=labels, dg_compute=True)
                auroc = auroc[2][0]
            else:
                raise Exception()
            return auroc

class PRROC_ROCAUC(PRROCScorer):
    def score(self, y_score: List[float], y_real: List[int]) -> float:
        from rpy2.rinterface_lib import openrlib
        with openrlib.rlock:
            pkg = import_PRROC()
            from rpy2.robjects.vectors import FloatVector
            labels = FloatVector(y_real)
            scores = FloatVector(y_score)
            auroc = pkg.roc_curve(scores, weights_class0=labels)
            auroc = auroc[1][0]
            return auroc

@dataclass
class ScorerInfo:
    name: str
    alias: str = ""
    params: dict = field(default_factory=dict)

    @classmethod
    def from_dict(cls, dt: dict):
        return cls(**dt)

    def __attrs_post_init__(self):
        if not self.alias:
            self.alias = self.name
    
    def make(self):
        if self.name == "scikit_rocauc":
            return SklearnROCAUC(self.alias)
        elif self.name == "scikit_prauc":
            return SklearnPRAUC(self.alias)
        elif self.name == "prroc_rocauc":
            return PRROC_ROCAUC(self.alias)
        elif self.name == "prroc_prauc":
            tp = self.params.get("type")
            if tp is None:
                raise Exception("type must be specified for prauc scorer from PRROC package")
            tp = tp.lower()
            return PRROC_PRAUC(self.alias, tp)
        elif self.name == "constant_scorer":
            cons = self.params.get("cons")
            if cons is None:
                raise Exception("cons must be specified for constant scorer")
            cons = float(cons)
            return ConstantScorer(self.alias, cons)
        raise Exception(f"Wrong scorer: {self.name}")
    
    def to_dict(self) -> dict:
        dt = {}
        dt['name'] = self.name
        dt['alias'] = self.alias
        dt['params'] = self.params
        return dt
    
import scorer_module


score_calc_rocauc = ScorerInfo("prroc_rocauc", "rocauc").make()

score_calc_prauc = ScorerInfo("prroc_prauc", "prauc", params={"type": "integral"}).make()


In [ ]:
pos_class = size_of_set
neg_class = pos_class * 100
N_features = 1000         
organism = "HUMAN_MOUSE"
n_jobs = 40
plot_slim = True
scale_data = True
print_GC = False
verbose = True


today_date = "NN" #"15.04.2024"
    
def scrambled(orig):
    dest = orig[:]
    shuffle(dest)
    return dest

for mode in ["mono", "di", "mono_di"]: # selecting which PWMs we are going to use
    for model_name in ["RandomForestClassifier"]: #["BaggingClassifier_LogisticRegression", "XGBClassifier", "RandomForestClassifier", "LogisticRegression", "BaggingClassifier_XGBClassifier"]:  #"RandomForestClassifier_M_only_and_all_in_SLIM"]: #, "LogisticRegression", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression" "XGBClassifier"]: # setting up models
        sys.setrecursionlimit = 10**3 # for deeper recursion

        # The module for class creation TFn - 1, Others - 0       
        def group_selection_GC(flag, TF_name, df, pos_class, neg_class, new_dir_name, outfile_name_no_NF, mode):
            df = df.dropna(subset=[1])

            df_group = df.copy()
            if verbose == True:
                print(df_group)
            print(" ")
            print("Group_selection is running ...")

            if TF_name.split("_")[1] == "HUMAN":
                print("HUMAN")
                columns_name = df_group.columns[3] # parsing file names
                train_pos = [x == TF_name for x in df_group[columns_name]] # selecting by TF name
                train_pos = df_group[train_pos] # selecting positive set
 
                if verbose == True:
                    print("Train positives length before size selection:", train_pos.shape[0])
                if train_pos.shape[0] > pos_class: # Size check
                    df1 = train_pos.sample(n=pos_class, replace=False, random_state=0)
                else:
                    df1 = train_pos
                if verbose == True:
                    print("Train positives length after size selection:", df1.shape[0])
                arr1 = [1] * df1.shape[0]
                df1 = df1.assign(ind=arr1) # adding class flags

                isfile = [f for f in os.listdir(new_dir_name) if os.path.splitext(f)[1] == '.fasta']
                if f"{mode}_neg_id_out_HUMAN_{flag}.csv" not in new_dir_name:
                    if f"out_HUMAN_{flag}.fasta" not in isfile: # creafing fasta file with negative set
                        TF_mask = TF_family_dict.get(TF_name.split('_')[0], None) # parsing TF code in a dictionary
                        if TF_mask != None: # if we know TF family
                            if verbose == True:
                                print("Before family selection for " + TF_mask + " : ", df_group.shape[0])
                            df_mask = df_group[2] != float(TF_mask) # selecting other families' TFs to negative set
                            train_neg = df_group[df_mask]
                            if verbose == True:
                                print("After family selection:", train_neg.shape[0])
                        else:
                            train_neg = [x != TF_name for x in df_group[columns_name]] # or selecting everything if we know nothing
                            train_neg = df_group[train_neg]
    
                        df1[1].to_csv(new_dir_name + f"/{mode}_pos_id_HUMAN.csv", sep="\n", index=False) # writing ids for BiasAway
                        train_neg.to_csv(new_dir_name + f"/{mode}_neg_id_HUMAN.csv", sep="\n", index=False)
                        print("Saved files with ids for BiasAway")
    
                        # selecting fasta records by ids
                        line1 = f"{root}/seqtk/seqtk subseq {outfile_name_no_NF} {mode}_pos_id_HUMAN.csv > {mode}_pos_id_HUMAN.fasta; {root}/seqtk/seqtk subseq {outfile_name_no_NF} {mode}_neg_id_HUMAN.csv > {mode}_neg_id_HUMAN.fasta"
                        p = subprocess.Popen(line1, shell=True)
                        p.wait()
    
                        # running BiasAway and creating negative set
                        neg_size = str(int(neg_class/pos_class))
                        line2 = f"biasaway g -r BiasAway_results_HUMAN_{flag} -f {mode}_pos_id_HUMAN.fasta -b {mode}_neg_id_HUMAN.fasta -n {neg_size} -p BiasAway_results_HUMAN_{mode} -e 1 > out_HUMAN_{flag}.fasta 2> BiasAway_log_HUMAN.txt" 
                        p = subprocess.Popen(line2, shell=True)
                        p.wait()
                        print("BiasAway completed")
                        
                        # selecting ids
                        line3 = "awk 'sub(/^>/, \"\")' < out_HUMAN_"+flag+f".fasta > {mode}_neg_id_out_HUMAN_{flag}.csv"
                        p = subprocess.Popen(line3, shell=True)
                        p.wait()


                df2_names = pd.read_csv(new_dir_name + f"/{mode}_neg_id_out_HUMAN_{flag}.csv", header=None)
                df2_names_list = list(df2_names[0])
                if verbose == True:
                    print("After BiasAway negative set contains:", len(df2_names_list))

                train_neg_GC = df_group[df_group[1].isin(df2_names_list)]
                if verbose == True:
                    print("Train negative set after BiasAway:", train_neg_GC.shape[0])

                # trimming negative set
                if train_neg_GC.shape[0] > neg_class:
                    df2 = train_neg_GC.sample(n=neg_class, replace=False, random_state=0)
                else:
                    df2 = train_neg_GC
                
                if verbose == True:
                    print("Train negative set after filtration:", df2.shape[0])
                arr2 = [0] * df2.shape[0]
                df2 = df2.assign(ind=arr2)

                if neg_class == pos_class:
                    len_df1 = df1.shape[0] # equalizing sets
                    len_df2 = df2.shape[0]

                    if len_df1 != len_df2:
                        if len_df1 > len_df2:
                            df1 = df1.sample(n=len_df2, replace=False, random_state=0)
                        else:
                            df2 = df2.sample(n=len_df1, replace=False, random_state=0)

                if verbose == True:
                    print("After selection we have:", df1.shape[0], df2.shape[0])

                result = pd.concat([df1, df2])
                return result


            if TF_name.split("_")[1] == "MOUSE": # repeating the same for MOUSE
                print("MOUSE")
                columns_name = df_group.columns[3]
                train_pos = [x == TF_name for x in df_group[columns_name]]
                train_pos = df_group[train_pos]

                if verbose == True:
                    print("Train positives length before size selection:", train_pos.shape[0])
                if train_pos.shape[0] > pos_class:
                    df1 = train_pos.sample(n=pos_class, replace=False, random_state=0)
                else:
                    df1 = train_pos
                if verbose == True:
                    print("Train positives length after size selection:", df1.shape[0])
                arr1 = [1] * df1.shape[0]
                df1 = df1.assign(ind=arr1)

                isfile = [f for f in os.listdir(new_dir_name) if os.path.splitext(f)[1] == '.fasta']
                if  f"{mode}_neg_id_out_MOUSE_{flag}.csv" not in new_dir_name:
                    if f"out_MOUSE_{flag}.fasta" not in isfile:
    
                        TF_mask = TF_family_dict.get(TF_name.split('_')[0], None)
                        if TF_mask != None:
                            if verbose == True:
                                print("Before family selection for " + TF_mask + " : ", df_group.shape[0])
                            df_mask = df_group[2] != float(TF_mask)
                            train_neg = df_group[df_mask]
                            if verbose == True:
                                print("After family selection:", train_neg.shape[0])
                        else:
                            train_neg = [x != TF_name for x in df_group[columns_name]]
                            train_neg = df_group[train_neg]
    
                        df1[1].to_csv(new_dir_name + f"/{mode}_pos_id_MOUSE.csv", sep="\n", index=False) 
                        train_neg.to_csv(new_dir_name + f"/{mode}_neg_id_MOUSE.csv", sep="\n", index=False)
                        print("Saved files with ids for BiasAway")
    
                        line1 = f"{root}/seqtk/seqtk subseq {outfile_name_no_NF} {mode}_pos_id_MOUSE.csv > {mode}_pos_id_MOUSE.fasta; {root}/seqtk/seqtk subseq {outfile_name_no_NF} {mode}_neg_id_MOUSE.csv > {mode}_neg_id_MOUSE.fasta"
                        p = subprocess.Popen(line1, shell=True)
                        p.wait()
    
                        neg_size = str(int(neg_class/pos_class))
                        line2 = f"biasaway g -r BiasAway_results_MOUSE_{flag} -f {mode}_pos_id_MOUSE.fasta -b {mode}_neg_id_MOUSE.fasta -n {neg_size} -p BiasAway_results_MOUSE_{mode} -e 1 > out_MOUSE_{flag}.fasta 2> BiasAway_log_MOUSE.txt" 
                        p = subprocess.Popen(line2, shell=True)
                        p.wait()
    
                        print("BiasAway completed")
                        line3 = "awk 'sub(/^>/, \"\")' < out_MOUSE_"+flag+f".fasta > {mode}_neg_id_out_MOUSE_{flag}.csv"
                        p = subprocess.Popen(line3, shell=True)
                        p.wait()

                df2_names = pd.read_csv(new_dir_name + f"/{mode}_neg_id_out_MOUSE_{flag}.csv", header=None)

                df2_names_list = list(df2_names[0])
                if verbose == True:
                    print("After BiasAway negative set contains:", len(df2_names_list))

                train_neg_GC = df_group[df_group[1].isin(df2_names_list)]
                if verbose == True:
                    print("Train negative set after BiasAway:", train_neg_GC.shape[0])

                if train_neg_GC.shape[0] > neg_class:
                    df2 = train_neg_GC.sample(n=neg_class, replace=False, random_state=0)
                else:
                    df2 = train_neg_GC
                if verbose == True:
                    print("Train negative set after filtration:", df2.shape[0])
                arr2 = [0] * df2.shape[0]
                df2 = df2.assign(ind=arr2)

                if neg_class == pos_class:
                    len_df1 = df1.shape[0]
                    len_df2 = df2.shape[0]

                    if len_df1 != len_df2:
                        if len_df1 > len_df2:
                            df1 = df1.sample(n=len_df2, replace=False, random_state=0)
                        else:
                            df2 = df2.sample(n=len_df1, replace=False, random_state=0)

                if verbose == True:
                    print("After selection we have:", df1.shape[0], df2.shape[0])

                result = pd.concat([df1, df2])
                return result


        # building models
        def model_building(X_train, X_test, Y_train, Y_test, model_name): 

            print(" ")
            print("Model_building is running ...")

            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

            if model_name == "XGBClassifier": # selecting a model      
                MODEL = XGBClassifier(**{'colsample_bytree': 0.8, 'gamma': 0.3, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 0.01, 'subsample': 0.8}, n_jobs=n_jobs, random_state=0)
            if model_name == "XGBClassifier_scrambled":    
                MODEL = XGBClassifier(**{'colsample_bytree': 0.8, 'gamma': 0.3, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 0.01, 'subsample': 0.8}, n_jobs=n_jobs)
            if model_name == "LogisticRegression":
                MODEL = LogisticRegression(n_jobs=n_jobs, **{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'})
            if model_name == "RandomForestClassifier":
                MODEL = RandomForestClassifier(**{'max_depth': 6, 'max_samples': 0.8, 'n_estimators': 100}, n_jobs=n_jobs)
            if model_name == "BaggingClassifier":
                MODEL = BaggingClassifier(base_estimator=XGBClassifier(max_depth=2, n_estimators=100), max_samples=0.05, n_estimators=50, n_jobs=n_jobs, random_state=0)
            if model_name == "BaggingClassifier_XGBClassifier":
                MODEL = BaggingClassifier(base_estimator=XGBClassifier(**{'colsample_bytree': 0.8, 'gamma': 0.3, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 0.01, 'subsample': 0.8}), n_jobs=n_jobs)
            if model_name == "BaggingClassifier_LogisticRegression":
                MODEL = BaggingClassifier(base_estimator=LogisticRegression(**{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}), n_jobs=n_jobs, random_state=0)
            if model_name == "BaggingClassifier_RandomForestClassifier":
                MODEL = BaggingClassifier(base_estimator=RandomForestClassifier(**{'max_depth': 25, 'max_samples': 0.8, 'n_estimators': 300}), n_jobs=n_jobs, random_state=0)
            if model_name == "AdaBoostClassifier":
                MODEL = AdaBoostClassifier(n_estimators=100, random_state=0)

            MODEL.fit(X_train, Y_train)

            Y_train_predicted_proba = MODEL.predict_proba(X_train)[:, 1]
            if verbose == True:
                print("train roc_auc_score", roc_auc_score(y_score=Y_train_predicted_proba, y_true=Y_train))
                print("train  average_precision_score", average_precision_score(y_score=Y_train_predicted_proba, y_true=Y_train))
            return MODEL, Y_train_predicted_proba


        # The model for curves plotting
        def rocauc_plotting(pwm_names, X_train_H_full, X_test_H_full, Y_train_H_full, Y_test_H_full,
                            X_train_M_full, X_test_M_full, Y_train_M_full, Y_test_M_full,
                            X_train_H, X_test_H, Y_train_H, Y_test_H,
                            Y_train_M, Y_test_M,
                            Y_train_predicted_proba_H,
                            Y_test_predicted_proba_H,
                            Y_test_predicted_proba_M, 
                            training_obj, testing_obj, 
                            features_c, 
                            model_name, 
                            mtrx_mono, TF, plot_slim, MODEL, mtrx_di=0):




            print(" ")
            print("ROC_AUC_plotting is running ...")
            fig = matplotlib.pyplot.gcf()
            fig.set_size_inches(16, 10)
            linewidth = 5
            color_palette = sns.color_palette("tab10")

            # selecting the best PWM on a train set
            f_c_tr_h = 0
            best_pwm_q_list = []
            best_pwm_number_list = []
            for k in range(len(X_train_H_full.columns)):
                f = X_train_H_full.columns[k]
                fpr_train_H_PWM, tpr_train_H_PWM, thresholds = roc_curve(Y_train_H_full, X_train_H_full[f])
                best_pwm_q_list.append(score_calc_rocauc.score(X_train_H_full[f], Y_train_H_full))
                best_pwm_number_list.append(k)

            best_pwm_q_list_1, best_pwm_number_list1 = zip(*sorted(zip(best_pwm_q_list, best_pwm_number_list), reverse=True))
            #print(best_pwm_number_list, best_pwm_q_list)
            f_c_tr_h = best_pwm_number_list1[0]


            f_c_tr_h_pr = 0
            best_pwm_q_list = []
            best_pwm_number_list = []
            for k in range(len(X_train_H_full.columns)):
                f = X_train_H_full.columns[k]
                precision_test_H_m, recall_test_H_m, thresholds = precision_recall_curve(Y_train_H_full, X_train_H_full[f])
                best_pwm_q_list.append(score_calc_prauc.score(X_train_H_full[f], Y_train_H_full))
                best_pwm_number_list.append(k)

            best_pwm_q_list_2, best_pwm_number_list2 = zip(*sorted(zip(best_pwm_q_list, best_pwm_number_list), reverse=True))
            #print(best_pwm_number_list, best_pwm_q_list)
            f_c_tr_h_pr = best_pwm_number_list2[0]

    
            fpr_train_H_PWM, tpr_train_H_PWM, thresholds = roc_curve(Y_train_H_full, X_train_H_full[X_train_H_full.columns[f_c_tr_h]])
            roc_auc_train_H_PWM_mono = score_calc_rocauc.score(X_train_H_full[X_train_H_full.columns[f_c_tr_h]], Y_train_H_full)
            plt.plot(fpr_train_H_PWM, tpr_train_H_PWM, linewidth=linewidth, label=f'Train PWM {training_obj} ROC (by roc) (AUC = %0.3f)' % roc_auc_train_H_PWM_mono, color=color_palette[0])


            fpr_test_H_PWM, tpr_test_H_PWM, thresholds = roc_curve(Y_test_H_full, X_test_H_full[X_test_H_full.columns[f_c_tr_h]])
            roc_auc_test_H_PWM_mono = score_calc_rocauc.score(X_test_H_full[X_test_H_full.columns[f_c_tr_h]], Y_test_H_full)
            plt.plot(fpr_test_H_PWM, tpr_test_H_PWM, linewidth=linewidth,label=f'Validation PWM {training_obj} ROC (AUC = %0.3f)' % roc_auc_test_H_PWM_mono, color=color_palette[1])

            fpr_test_M_PWM, tpr_test_M_PWM, thresholds = roc_curve(Y_test_M_full, X_test_M_full[X_test_M_full.columns[f_c_tr_h]])
            roc_auc_test_M_PWM_mono = score_calc_rocauc.score(X_test_M_full[X_test_M_full.columns[f_c_tr_h]], Y_test_M_full)
            plt.plot(fpr_test_M_PWM, tpr_test_M_PWM, linewidth=linewidth,label=f'Validation PWM {testing_obj} ROC (AUC = %0.3f)' % roc_auc_test_M_PWM_mono, color=color_palette[2])

            if mtrx_di !=0:
                fpr_train_H_PWM_di, tpr_train_H_PWM_di, thresholds = roc_curve(Y_train_H_full, X_train_H_full[mtrx_di])
                roc_auc_train_H_PWM_di = score_calc_rocauc.score(X_train_H_full[mtrx_di], Y_train_H_full)

                fpr_test_H_PWM_di, tpr_test_H_PWM_di, thresholds = roc_curve(Y_test_H_full, X_test_H_full[mtrx_di])
                roc_auc_test_H_PWM_di = score_calc_rocauc.score(X_test_H_full[mtrx_di], Y_test_H_full)

                fpr_test_M_PWM_di, tpr_test_M_PWM_di, thresholds = roc_curve(Y_test_M_full, X_test_M_full[mtrx_di])
                roc_auc_test_M_PWM_di = score_calc_rocauc.score(X_test_M_full[mtrx_di], Y_test_M_full)
            else:
                roc_auc_train_H_PWM_di = 0
                roc_auc_test_H_PWM_di = 0
                roc_auc_test_M_PWM_di = 0

            ##
            if features_c == 1:
                DF_ROC_PLOTS = pd.DataFrame()
                DF_ROC_PLOTS["fpr"] = fpr_train_H_PWM
                DF_ROC_PLOTS["tpr"] = tpr_train_H_PWM
                DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_ROC_PLOTS_H_PWM_mono_di_train.csv", sep="\t", index=False) # записываю 

                DF_ROC_PLOTS = pd.DataFrame()
                DF_ROC_PLOTS["fpr"] = fpr_test_H_PWM
                DF_ROC_PLOTS["tpr"] = tpr_test_H_PWM
                DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_ROC_PLOTS_H_PWM_mono_di_test.csv", sep="\t", index=False) # записываю 

                DF_ROC_PLOTS = pd.DataFrame()
                DF_ROC_PLOTS["fpr"] = fpr_test_M_PWM
                DF_ROC_PLOTS["tpr"] = tpr_test_M_PWM
                DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_ROC_PLOTS_M_PWM_mono_di_test.csv", sep="\t", index=False) # записываю 

            ##    
            #base_fpr, mean_tprs, tprs_lower, tprs_upper, mean_auc_test_M, median_auc_test_M, std_auc_test_M = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_test_M), np.array(Y_test_M), model_name, "ROC")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M, Y_test_M)
            #plt.fill_between(base_fpr, tprs_lower, tprs_upper, alpha = 0.2, facecolor=color_palette[3])
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[3])

            DF_ROC_PLOTS = pd.DataFrame()
            DF_ROC_PLOTS["fpr"] = fpr_test_M
            DF_ROC_PLOTS["tpr"] = tpr_test_M
            DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_ROC_PLOTS_M_mono_di_test.csv", sep="\t", index=False) # записываю 


            ##
            #base_fpr, mean_tprs, tprs_lower, tprs_upper, mean_auc_train_H, median_auc_train_H, std_auc_train_H = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_train_H), np.array(Y_train_H), model_name, "ROC", "train")

            fpr_train_H, tpr_train_H, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H)
            roc_auc_train_H = score_calc_rocauc.score(Y_train_predicted_proba_H, Y_train_H)
            #plt.fill_between(base_fpr, tprs_lower, tprs_upper, alpha = 0.2, facecolor=color_palette[4])
            plt.plot(fpr_train_H, tpr_train_H, linewidth=linewidth,label=f'Train {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_train_H), color=color_palette[4])

            DF_ROC_PLOTS = pd.DataFrame()
            DF_ROC_PLOTS["fpr"] = fpr_train_H
            DF_ROC_PLOTS["tpr"] = tpr_train_H
            DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_ROC_PLOTS_H_mono_di_train.csv", sep="\t", index=False) # записываю 

            ##
            #base_fpr, mean_tprs, tprs_lower, tprs_upper, mean_auc_test_H, median_auc_test_H, std_auc_test_H = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_test_H), np.array(Y_test_H), model_name, "ROC")

            fpr_test_H, tpr_test_H, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H)
            roc_auc_test_H = score_calc_rocauc.score(Y_test_predicted_proba_H, Y_test_H)
            #plt.fill_between(base_fpr, tprs_lower, tprs_upper, alpha = 0.2, facecolor=color_palette[5])
            plt.plot(fpr_test_H, tpr_test_H, linewidth=linewidth,label=f'Validation {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_H), color=color_palette[5])

            DF_ROC_PLOTS = pd.DataFrame()
            DF_ROC_PLOTS["fpr"] = fpr_test_H
            DF_ROC_PLOTS["tpr"] = tpr_test_H
            DF_ROC_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_ROC_PLOTS_H_mono_di_test.csv", sep="\t", index=False) # записываю 

            sns.set_context("paper", font_scale=3)
            plt.rc('legend',fontsize=12) # using a size in points
            plt.plot([0, 1], [0, 1], 'k--')  # ideal classifier
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.0])
            plt.xlabel('False Positive Rate', fontsize=25)
            plt.ylabel('True Positive Rate', fontsize=25)

            if features_c == 1:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} feature'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower right")
                plt.savefig('ROC_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_feature_{mode}.pdf'.format(mode=mode,model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)
            else:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} features'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower right")
                plt.savefig('ROC_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_features_{mode}.pdf'.format(mode=mode,model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)
            plt.show()
            plt.close()

            print(" ")
            print("PR_plotting is running ...")
            fig = matplotlib.pyplot.gcf()
            fig.set_size_inches(16, 10)

            ##
            precision_train_H_PWM, recall_train_H_PWM, thresholds = precision_recall_curve(Y_train_H_full, X_train_H_full[X_train_H_full.columns[f_c_tr_h_pr]])
            pr_auc_train_H_PWM_mono = score_calc_prauc.score(X_train_H_full[X_train_H_full.columns[f_c_tr_h_pr]], Y_train_H_full)
            plt.plot(recall_train_H_PWM, precision_train_H_PWM, linewidth=linewidth,label=f'Train PWM {training_obj} PR (by pr) (AUC = %0.3f)' % pr_auc_train_H_PWM_mono, color=color_palette[0])

        #     precision_train_H_PWM, recall_train_H_PWM, thresholds = precision_recall_curve(Y_train_H, X_train_H[X_train_H.columns[f_c_tr_h]])
        #     pr_auc_train_H_PWM_mono = score_calc_prauc(Y_train_H, X_train_H[X_train_H.columns[f_c_tr_h]])
        #     plt.plot(recall_train_H_PWM, precision_train_H_PWM, linewidth=linewidth,label=f'Train PWM {training_obj} PR (by roc) (AUC = %0.3f)' % pr_auc_train_H_PWM_mono, color=color_palette[0])

            precision_test_H_PWM, recall_test_H_PWM, thresholds = precision_recall_curve(Y_test_H_full, X_test_H_full[X_test_H_full.columns[f_c_tr_h_pr]])
            pr_auc_test_H_PWM_mono = score_calc_prauc.score(X_test_H_full[X_test_H_full.columns[f_c_tr_h_pr]], Y_test_H_full)
            plt.plot(recall_test_H_PWM, precision_test_H_PWM, linewidth=linewidth,label=f'Validation PWM {training_obj} PR (AUC = %0.3f)' % pr_auc_test_H_PWM_mono, color=color_palette[1])

            precision_test_M_PWM, recall_test_M_PWM, thresholds = precision_recall_curve(Y_test_M_full, X_test_M_full[X_test_M_full.columns[f_c_tr_h_pr]])
            pr_auc_test_M_PWM_mono = score_calc_prauc.score(X_test_M_full[X_test_M_full.columns[f_c_tr_h_pr]], Y_test_M_full)
            plt.plot(recall_test_M_PWM, precision_test_M_PWM, linewidth=linewidth,label=f'Validation PWM {testing_obj} PR (AUC = %0.3f)' % pr_auc_test_M_PWM_mono, color=color_palette[2])

            if mtrx_di !=0:
                precision_train_H_PWM_di, recall_train_H_PWM_di, thresholds = precision_recall_curve(Y_train_H_full, X_train_H_full[mtrx_mono])
                pr_auc_train_H_PWM_di = score_calc_prauc.score(X_train_H_full[mtrx_mono], Y_train_H_full)

                precision_test_H_PWM_di, recall_test_H_PWM_di, thresholds = precision_recall_curve(Y_test_H_full, X_test_H_full[mtrx_mono])
                pr_auc_test_H_PWM_di = score_calc_prauc.score(X_test_H_full[mtrx_mono], Y_test_H_full)

                precision_test_M_PWM_di, recall_test_M_PWM_di, thresholds = precision_recall_curve(Y_test_M_full, X_test_M_full[mtrx_mono])
                pr_auc_test_M_PWM_di = score_calc_prauc.score(X_test_M_full[mtrx_mono], Y_test_M_full)
            else:
                pr_auc_train_H_PWM_di = 0
                pr_auc_test_H_PWM_di = 0
                pr_auc_test_M_PWM_di = 0
            ##
            if features_c == 1:
                DF_PR_PLOTS = pd.DataFrame()
                DF_PR_PLOTS["precision"] = precision_train_H_PWM
                DF_PR_PLOTS["recall"] = recall_train_H_PWM
                DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_PR_PLOTS_H_PWM_mono_di_train.csv", sep="\t", index=False) # записываю 

                DF_PR_PLOTS = pd.DataFrame()
                DF_PR_PLOTS["precision"] = precision_test_H_PWM
                DF_PR_PLOTS["recall"] = recall_test_H_PWM
                DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_PR_PLOTS_H_PWM_mono_di_test.csv", sep="\t", index=False) # записываю 

                DF_PR_PLOTS = pd.DataFrame()
                DF_PR_PLOTS["precision"] = precision_test_M_PWM
                DF_PR_PLOTS["recall"] = recall_test_M_PWM
                DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) +"_DF_PR_PLOTS_M_PWM_mono_di_test.csv", sep="\t", index=False) # записываю 


            # base_recall, mean_precs, precs_lower, precs_upper, mean_pr_auc_test_M, median_pr_auc_test_M, std_pr_auc_test_M = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_test_M), np.array(Y_test_M), model_name, "PR")
          
            precision_test_M, recall_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M)
            pr_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M, Y_test_M)
            #plt.fill_between(base_recall, precs_lower, precs_upper, alpha = 0.2, facecolor=color_palette[3])
            plt.plot(recall_test_M, precision_test_M, linewidth=linewidth,label='Validation MOUSE PR (AUC = %0.2f)' % (pr_auc_test_M), color=color_palette[3])


            DF_PR_PLOTS = pd.DataFrame()
            DF_PR_PLOTS["precision"] = precision_test_M
            DF_PR_PLOTS["recall"] = recall_test_M
            DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_PR_PLOTS_M_mono_di_test.csv", sep="\t", index=False) # записываю 

            ##
            #base_recall, mean_precs, precs_lower, precs_upper, mean_pr_auc_train_H, median_pr_auc_train_H, std_pr_auc_train_H = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_train_H), np.array(Y_train_H), model_name, "PR", "train")
            precision_train_H, recall_train_H, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H)
            pr_auc_train_H = score_calc_prauc.score(Y_train_predicted_proba_H, Y_train_H)
            #plt.fill_between(base_recall, precs_lower, precs_upper, alpha = 0.2, facecolor=color_palette[4])
            plt.plot(recall_train_H, precision_train_H, linewidth=linewidth,label='Train HUMAN PR (AUC = %0.2f)' % (pr_auc_train_H), color=color_palette[4])

            DF_PR_PLOTS = pd.DataFrame()
            DF_PR_PLOTS["precision"] = precision_train_H
            DF_PR_PLOTS["recall"] = recall_train_H
            DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_PR_PLOTS_H_mono_di_train.csv", sep="\t", index=False) # записываю 

            ##
            #base_recall, mean_precs, precs_lower, precs_upper, mean_pr_auc_test_H, median_pr_auc_test_H, std_pr_auc_test_H = conf_int_plot(MODEL, np.array(X_train_H), np.array(Y_train_H), np.array(X_test_H), np.array(Y_test_H), model_name, "PR")
            precision_test_H, recall_test_H, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H)
            pr_auc_test_H = score_calc_prauc.score(Y_test_predicted_proba_H, Y_test_H)
            #plt.fill_between(base_recall, precs_lower, precs_upper, alpha = 0.2, facecolor=color_palette[5])
            plt.plot(recall_test_H, precision_test_H, linewidth=linewidth,label='Validation HUMAN PR (AUC = %0.2f)' % (pr_auc_test_H), color=color_palette[5])

            DF_PR_PLOTS = pd.DataFrame()
            DF_PR_PLOTS["precision"] = precision_test_H
            DF_PR_PLOTS["recall"] = recall_test_H
            DF_PR_PLOTS.to_csv(new_dir_name + "/" + TF + "_" + model_name + "_" + str(features_c) + "_DF_PR_PLOTS_H_mono_di_test.csv", sep="\t", index=False) # записываю 

            sns.set_context("paper", font_scale=3)
            plt.rc('legend',fontsize=12)
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.0])
            plt.xlabel('Recall', fontsize=25)
            plt.ylabel('Precision', fontsize=25)

            if features_c == 1:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} feature'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower left")
                plt.savefig('PR_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_feature_{mode}.pdf'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)
                line_f = f'echo features_c roc_auc_train_H_PWM_mono roc_auc_train_H_PWM_di roc_auc_test_H_PWM_mono roc_auc_test_H_PWM_di roc_auc_test_M_PWM_mono roc_auc_test_M_PWM_di ' \
                         f'roc_auc_train_H ' \
                         f'roc_auc_test_H ' \
                         f'roc_auc_test_M ' \
                         f'pr_auc_train_H ' \
                         f'pr_auc_test_H ' \
                         f'pr_auc_test_M ' \
                         f'pr_auc_train_H_PWM_mono pr_auc_train_H_PWM_di pr_auc_test_H_PWM_mono pr_auc_test_H_PWM_di pr_auc_test_M_PWM_mono pr_auc_test_M_PWM_di ' \
                         f' >> {basicdir}/outputdir/{TF}/new_log_roc_pr_HUMAN_MOUSE_{mode}_{model_name}_{mode}_{today_date}.txt'
                p = subprocess.Popen(line_f, shell=True)
                p.wait()

            else:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} features'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower left")
                plt.savefig('PR_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_features_{mode}.pdf'.format(mode=mode,model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)


            line_f = f'echo {features_c} {roc_auc_train_H_PWM_mono} {roc_auc_train_H_PWM_di} {roc_auc_test_H_PWM_mono} {roc_auc_test_H_PWM_di} {roc_auc_test_M_PWM_mono} {roc_auc_test_M_PWM_di} ' \
                         f'{roc_auc_train_H} ' \
                         f'{roc_auc_test_H} ' \
                         f'{roc_auc_test_M} ' \
                         f'{pr_auc_train_H} ' \
                         f'{pr_auc_test_H} ' \
                         f'{pr_auc_test_M} ' \
                         f'{pr_auc_train_H_PWM_mono} {pr_auc_train_H_PWM_di} {pr_auc_test_H_PWM_mono} {pr_auc_test_H_PWM_di} {pr_auc_test_M_PWM_mono} {pr_auc_test_M_PWM_di} ' \
                     f' >> {basicdir}/outputdir/{TF}/new_log_roc_pr_HUMAN_MOUSE_{mode}_{model_name}_{mode}_{today_date}.txt'
            print(line_f)
            p = subprocess.Popen(line_f, shell=True)
            p.wait()
            plt.show()

        def rocauc_plotting2(pwm_names, X_train_H_full, X_test_H_full, Y_train_H_full, Y_test_H_full,
            X_train_M_full, X_test_M_full, Y_train_M_full, Y_test_M_full,
            X_train_H, X_test_H, Y_train_H, Y_test_H,
            X_train_M, X_test_M, Y_train_M, Y_test_M,
            Y_train_predicted_proba_H_full,
            Y_train_predicted_proba_H,
            Y_train_predicted_proba_H_munk,
            Y_train_predicted_proba_H1,
            Y_train_predicted_proba_H5,
            Y_train_predicted_proba_H015,
            Y_test_predicted_proba_H_full,
            Y_test_predicted_proba_H,
            Y_test_predicted_proba_H_munk,
            Y_test_predicted_proba_H1,
            Y_test_predicted_proba_H5,
            Y_test_predicted_proba_H015,
            Y_test_predicted_proba_M_full,
            Y_test_predicted_proba_M,
            Y_test_predicted_proba_M_munk,
            Y_test_predicted_proba_M1,
            Y_test_predicted_proba_M5,
            Y_test_predicted_proba_M015,
            training_obj, testing_obj, 
            features_c, 
            model_name, 
            mtrx_mono, TF, plot_slim, MODEL_full, MODEL, MODEL_munk, MODEL1, MODEL5, MODEL015, mtrx_di):


            print(" ")
            print("ROC_AUC_plotting is running ...")
            sns.set_style(style='white') 
            fig = matplotlib.pyplot.gcf()
            fig.set_size_inches(16, 10)
            linewidth = 5
            color_palette = sns.color_palette("tab20")
            ##

            f_c_tr_h = 0
            best_pwm_q_list = []
            best_pwm_number_list = []
            for k in range(len(X_train_H_full.columns)):
                f = X_train_H_full.columns[k]
                fpr_train_H_PWM, tpr_train_H_PWM, thresholds = roc_curve(Y_train_H_full, X_train_H_full[f])
                best_pwm_q_list.append(score_calc_rocauc.score(X_train_H_full[f], Y_train_H_full))
                best_pwm_number_list.append(k)

            best_pwm_q_list_1, best_pwm_number_list1 = zip(*sorted(zip(best_pwm_q_list, best_pwm_number_list), reverse=True))
            #print(best_pwm_number_list, best_pwm_q_list)
            f_c_tr_h = best_pwm_number_list1[0]

            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(f_c_tr_h) + "\t")
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(pwm_names[f_c_tr_h]) + "\t")
                
                
            f_c = f_c_tr_h

            f_c_tr_h_pr = 0
            best_pwm_q_list = []
            best_pwm_number_list = []
            for k in range(len(X_train_H_full.columns)):
                f = X_train_H_full.columns[k]
                precision_test_H_m, recall_test_H_m, thresholds = precision_recall_curve(Y_train_H_full, X_train_H_full[f])
                best_pwm_q_list.append(score_calc_prauc.score(X_train_H_full[f], Y_train_H_full))
                best_pwm_number_list.append(k)

            best_pwm_q_list_2, best_pwm_number_list2 = zip(*sorted(zip(best_pwm_q_list, best_pwm_number_list), reverse=True))
            #print(best_pwm_number_list, best_pwm_q_list)
            f_c_tr_h_pr = best_pwm_number_list2[0]

            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(f_c_tr_h_pr) + "\t")
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(pwm_names[f_c_tr_h_pr]) + "\t")
                
                            

            fpr_train_H_PWM_m, tpr_train_H_PWM_m, thresholds = roc_curve(Y_train_H_full, X_train_H_full[X_train_H_full.columns[f_c_tr_h]])
            roc_auc_train_H_PWM_mono_m = score_calc_rocauc.score(X_train_H_full[X_train_H_full.columns[f_c_tr_h]], Y_train_H_full)
            plt.plot(fpr_train_H_PWM_m, tpr_train_H_PWM_m, linewidth=linewidth, label=f'Train PWM HUMAN ROC (AUC = %0.3f)' % roc_auc_train_H_PWM_mono_m, color=color_palette[0])


            fpr_test_H_PWM_m, tpr_test_H_PWM_m, thresholds = roc_curve(Y_test_H_full, X_test_H_full[X_test_H_full.columns[f_c_tr_h]]) # there was train bug here
            roc_auc_test_H_PWM_mono_m = score_calc_rocauc.score(X_test_H_full[X_test_H_full.columns[f_c_tr_h]], Y_test_H_full)
            plt.plot(fpr_test_H_PWM_m, tpr_test_H_PWM_m, linewidth=linewidth,label=f'Validation PWM HUMAN ROC (AUC = %0.3f)' % roc_auc_test_H_PWM_mono_m, color=color_palette[1])


            fpr_test_M_PWM_m, tpr_test_M_PWM_m, thresholds = roc_curve(Y_test_M_full, X_test_M_full[X_test_M_full.columns[f_c_tr_h]])
            roc_auc_test_M_PWM_mono_m = score_calc_rocauc.score(X_test_M_full[X_test_M_full.columns[f_c_tr_h]], Y_test_M_full)

            plt.plot(fpr_test_M_PWM_m, tpr_test_M_PWM_m, linewidth=linewidth,label=f'Validation PWM MOUSE ROC (AUC = %0.3f)' % roc_auc_test_M_PWM_mono_m, color=color_palette[2])

            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_train_H_PWM_mono_m) + "\t")

            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_H_PWM_mono_m) + "\t")

            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M_PWM_mono_m) + "\t")


            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_train["ind"]), list(result_H_train["pred0"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_train["pred0"]), list(result_H_train["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=0 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_train["ind"]), list(result_H_train["pred1"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_train["pred1"]), list(result_H_train["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_train["ind"]), list(result_H_train["pred5"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_train["pred5"]), list(result_H_train["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=-5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_train["ind"]), list(result_H_train["pred7"]))
            # roc_auc_test_M = score_calc_rocauc.score(list(result_H_train["pred7"]), list(result_H_train["ind"]))
            # plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=-7 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_train["ind"]), list(result_H_train[int(lastcol)]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_train[int(lastcol)]), list(result_H_train["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[3])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_control["ind"]), list(result_H_control["pred0"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_control["pred0"]), list(result_H_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=0 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_control["ind"]), list(result_H_control["pred1"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_control["pred1"]), list(result_H_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_control["ind"]), list(result_H_control["pred5"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_control["pred5"]), list(result_H_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_control["ind"]), list(result_H_control["pred7"]))
            # roc_auc_test_M = score_calc_rocauc.score(list(result_H_control["pred7"]), list(result_H_control["ind"]))
            # plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-7 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_H_control["ind"]), list(result_H_control[int(lastcol)]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_H_control[int(lastcol)]), list(result_H_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[8])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_M_control["ind"]), list(result_M_control["pred0"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_M_control["pred0"]), list(result_M_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=0 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_M_control["ind"]), list(result_M_control["pred1"]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_M_control["pred1"]), list(result_M_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=1 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_M_control["ind"]), list(result_M_control["pred5"])) # there was pred1 bug here
            roc_auc_test_M = score_calc_rocauc.score(list(result_M_control["pred5"]), list(result_M_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-5 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_M_control["ind"]), list(result_M_control["pred7"]))
            # roc_auc_test_M = score_calc_rocauc.score(list(result_M_control["pred7"]), list(result_M_control["ind"]))
            # plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-7 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(list(result_M_control["ind"]), list(result_M_control[int(lastcol)]))
            roc_auc_test_M = score_calc_rocauc.score(list(result_M_control[int(lastcol)]), list(result_M_control["ind"]))
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[11])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H_full, Y_train_predicted_proba_H_full)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H_full, Y_train_H_full)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train full {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[12])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H, Y_train_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[12])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H_munk)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H_munk, Y_train_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[13])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H1)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H1, Y_train_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM 1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[14])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H5)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H5, Y_train_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM -5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[15])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_train_H, Y_train_predicted_proba_H015)
            roc_auc_test_M = score_calc_rocauc.score(Y_train_predicted_proba_H015, Y_train_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM 1,-5 +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[16])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H_full, Y_test_predicted_proba_H_full)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H_full, Y_test_H_full)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation full {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[1])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H, Y_test_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[1])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H_munk)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H_munk, Y_test_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[2])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H1)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H1, Y_test_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[3])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H5)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H5, Y_test_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM -5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_H, Y_test_predicted_proba_H015)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_H015, Y_test_H)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1,-5 +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M_full, Y_test_predicted_proba_M_full)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M_full, Y_test_M_full)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation full {model_name}_{mode} MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M, Y_test_M)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M_munk)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M_munk, Y_test_M)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M1)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M1, Y_test_M)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[8])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M5)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M5, Y_test_M)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM -5 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = roc_curve(Y_test_M, Y_test_predicted_proba_M015)
            roc_auc_test_M = score_calc_rocauc.score(Y_test_predicted_proba_M015, Y_test_M)
            plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1,-5 +diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\n")

            sns.set_context("paper", font_scale=3)
            plt.rc('legend',fontsize=12)
            plt.plot([0, 1], [0, 1], 'k--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.0])
            plt.xlabel('False Positive Rate', fontsize=25)
            plt.ylabel('True Positive Rate', fontsize=25)

            if features_c == 1:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} feature'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower right")
                plt.savefig('ROC_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_feature_{mode}_SLIM.pdf'.format(mode=mode,model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)
            else:
                plt.title('{model_name}_{mode} on besthits. {training_obj}/{testing_obj}. {N} features'.format(mode=mode, model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), fontsize=20)
                plt.legend(loc="lower right")
                plt.savefig('ROC_AUC_{model_name}_{mode}_Train_{training_obj}_Test_{testing_obj}_{N}_features_{mode}_SLIM.pdf'.format(mode=mode,model_name=model_name, N=features_c, training_obj=training_obj, testing_obj=testing_obj), dpi=100)
            plt.show()
            plt.close()

            fpr_train_H_PWM_m, tpr_train_H_PWM_m, thresholds = precision_recall_curve(Y_train_H_full, X_train_H_full[X_train_H_full.columns[f_c_tr_h]])
            roc_auc_train_H_PWM_mono_m = score_calc_prauc.score(X_train_H_full[X_train_H_full.columns[f_c_tr_h]], Y_train_H_full)
            #plt.plot(fpr_train_H_PWM_m, tpr_train_H_PWM_m, linewidth=linewidth, label=f'Train PWM HUMAN ROC (AUC = %0.3f)' % roc_auc_train_H_PWM_mono_m, color=color_palette[0])


            fpr_test_H_PWM_m, tpr_test_H_PWM_m, thresholds = precision_recall_curve(Y_test_H_full, X_test_H_full[X_test_H_full.columns[f_c_tr_h]]) # there was train bug here
            roc_auc_test_H_PWM_mono_m = score_calc_prauc.score(X_test_H_full[X_test_H_full.columns[f_c_tr_h]], Y_test_H_full)
            #plt.plot(fpr_test_H_PWM_m, tpr_test_H_PWM_m, linewidth=linewidth,label=f'Validation PWM HUMAN ROC (AUC = %0.3f)' % roc_auc_test_H_PWM_mono_m, color=color_palette[1])


            fpr_test_M_PWM_m, tpr_test_M_PWM_m, thresholds = precision_recall_curve(Y_test_M_full, X_test_M_full[X_test_M_full.columns[f_c_tr_h]])
            roc_auc_test_M_PWM_mono_m = score_calc_prauc.score(X_test_M_full[X_test_M_full.columns[f_c_tr_h]], Y_test_M_full)

            #plt.plot(fpr_test_M_PWM_m, tpr_test_M_PWM_m, linewidth=linewidth,label=f'Validation PWM MOUSE ROC (AUC = %0.3f)' % roc_auc_test_M_PWM_mono_m, color=color_palette[2])

            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_train_H_PWM_mono_m) + "\t")

            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_H_PWM_mono_m) + "\t")

            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M_PWM_mono_m) + "\t")


            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_train["ind"]), list(result_H_train["pred0"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_train["pred0"]), list(result_H_train["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=0 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_train["ind"]), list(result_H_train["pred1"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_train["pred1"]), list(result_H_train["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_train["ind"]), list(result_H_train["pred5"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_train["pred5"]), list(result_H_train["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=-5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_train["ind"]), list(result_H_train["pred7"]))
            # roc_auc_test_M = score_calc_prauc.score(list(result_H_train["pred7"]), list(result_H_train["ind"]))
            # #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train SLIM m=-7 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_train["ind"]), list(result_H_train[int(lastcol)]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_train[int(lastcol)]), list(result_H_train["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[3])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_control["ind"]), list(result_H_control["pred0"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_control["pred0"]), list(result_H_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=0 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_control["ind"]), list(result_H_control["pred1"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_control["pred1"]), list(result_H_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_control["ind"]), list(result_H_control["pred5"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_control["pred5"]), list(result_H_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_control["ind"]), list(result_H_control["pred7"]))
            # roc_auc_test_M = score_calc_prauc.score(list(result_H_control["pred7"]), list(result_H_control["ind"]))
            # #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-7 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_H_control["ind"]), list(result_H_control[int(lastcol)]))
            roc_auc_test_M = score_calc_prauc.score(list(result_H_control[int(lastcol)]), list(result_H_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[8])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_M_control["ind"]), list(result_M_control["pred0"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_M_control["pred0"]), list(result_M_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=0 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_M_control["ind"]), list(result_M_control["pred1"]))
            roc_auc_test_M = score_calc_prauc.score(list(result_M_control["pred1"]), list(result_M_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=1 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_M_control["ind"]), list(result_M_control["pred5"])) # there was pred1 bug here
            roc_auc_test_M = score_calc_prauc.score(list(result_M_control["pred5"]), list(result_M_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-5 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            # fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_M_control["ind"]), list(result_M_control["pred7"]))
            # roc_auc_test_M = score_calc_prauc.score(list(result_M_control["pred7"]), list(result_M_control["ind"]))
            # #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation SLIM m=-7 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            # with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
            #     log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(list(result_M_control["ind"]), list(result_M_control[int(lastcol)]))
            roc_auc_test_M = score_calc_prauc.score(list(result_M_control[int(lastcol)]), list(result_M_control["ind"]))
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[11])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H_full, Y_train_predicted_proba_H_full)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H_full, Y_train_H_full)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train full {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[12])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H, Y_train_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[12])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H_munk)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H_munk, Y_train_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[13])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H1)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H1, Y_train_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM 1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[14])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H5)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H5, Y_train_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM -5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[15])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_train_H, Y_train_predicted_proba_H015)
            roc_auc_test_M = score_calc_prauc.score(Y_train_predicted_proba_H015, Y_train_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Train {model_name}_{mode} +SLIM 1,-5 +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[16])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H_full, Y_test_predicted_proba_H_full)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H_full, Y_test_H_full)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation full {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[1])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H, Y_test_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[1])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H_munk)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H_munk, Y_test_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[2])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H1)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H1, Y_test_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[3])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H5)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H5, Y_test_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM -5 HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[4])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_H, Y_test_predicted_proba_H015)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_H015, Y_test_H)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1,-5 +diChIPMunk HUMAN ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[5])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M_full, Y_test_predicted_proba_M_full)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M_full, Y_test_M_full)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation full {model_name}_{mode} MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")
                
            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M, Y_test_M)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[6])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M_munk)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M_munk, Y_test_M)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[7])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M1)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M1, Y_test_M)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[8])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M5)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M5, Y_test_M)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM -5 MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[9])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\t")

            fpr_test_M, tpr_test_M, thresholds = precision_recall_curve(Y_test_M, Y_test_predicted_proba_M015)
            roc_auc_test_M = score_calc_prauc.score(Y_test_predicted_proba_M015, Y_test_M)
            #plt.plot(fpr_test_M, tpr_test_M, linewidth=linewidth,label=f'Validation {model_name}_{mode} +SLIM 1,-5 +diChIPMunk MOUSE ROC (AUC = %0.2f)' % (roc_auc_test_M), color=color_palette[10])
            with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                log_f.write(str(roc_auc_test_M) + "\n")


        def conf_int_plot(MODEL, X,y,X2,y2, model_name, flag, train_flaf="", n_splits = 7):
            kf = StratifiedKFold(n_splits = n_splits, shuffle = True, random_state= 0)

            tprs = []
            fprs = []
            precs = []
            recs = []
            roc_aucs = []
            pr_aucs = []

            if verbose == True:
                print("N - ", len(y)//n_splits)
            base_fpr = np.linspace(0, 1, len(y)//n_splits)
            base_recall = np.linspace(0, 1, len(y)//n_splits)

            for (train, test), (train2, test2) in zip(kf.split(X,y), kf.split(X2,y2)):
                if train_flaf == "train":
                    y_score = MODEL.predict_proba(X[test])
                    fpr, tpr, _ = roc_curve(y[test], y_score[:, 1])
                    precision, recall, _ = precision_recall_curve(y[test], y_score[:, 1])
                else:
                    MODEL = MODEL.fit(X[train], y[train])
                    y_score = MODEL.predict_proba(X2[test2])
                    fpr, tpr, _ = roc_curve(y2[test2], y_score[:, 1])
                    precision, recall, _ = precision_recall_curve(y2[test2], y_score[:, 1])
                roc_auc = auc(fpr, tpr)
                if verbose == True:
                    print("roc_auc:", roc_auc)
                roc_aucs.append(roc_auc)
                tpr = interp(base_fpr, fpr, tpr)
                tpr[0] = 0.0
                tprs.append(tpr)


                pr_auc = auc(recall, precision)
                if verbose == True:
                    print("pr_auc:", pr_auc)
                pr_aucs.append(pr_auc)
                precision = interp(base_recall, recall, precision)
                precision[0] = 0.0
                precs.append(precision)

            if flag == "ROC":
                mean_roc_aucs = np.mean(roc_aucs)
                median_roc_aucs = np.median(roc_aucs)
                std_roc_aucs = np.std(roc_aucs)

                tprs = np.array(tprs)
                mean_tprs = tprs.mean(axis=0)
                std_tprs = tprs.std(axis=0)
                tprs_upper = np.minimum(mean_tprs + std_tprs, 1)
                tprs_lower = mean_tprs - std_tprs
                return base_fpr, mean_tprs, tprs_lower, tprs_upper, mean_roc_aucs, median_roc_aucs, std_roc_aucs

            ###
            if flag == "PR":
                mean_pr_auc = np.mean(pr_aucs)
                median_pr_aucs = np.median(pr_aucs)
                std_pr_aucs = np.std(pr_aucs)

                precs = np.array(precs)
                mean_precs = precs.mean(axis=0)
                std_precs = precs.std(axis=0)
                precs_upper = np.minimum(mean_precs + std_precs, 1)
                precs_lower = mean_precs - std_precs
                return base_recall, mean_precs, precs_lower, precs_upper, mean_pr_auc, median_pr_aucs, std_pr_aucs


        def Scale_transform(X_test_df, scale_data=scale_data):
            if scale_data == True:
                X_test_df_out = X_test_df.copy()
                scaler = StandardScaler()
                X_test_df_out = pd.DataFrame(scaler.fit_transform(X_test_df_out), columns = X_test_df_out.columns)
                return X_test_df_out
            else:
                X_test_df_out = X_test_df.copy()
                return X_test_df_out



        TF_сalc = 0
        for TF in ["ANDR_HUMAN", "ESR1_HUMAN", "RXRA_HUMAN"]: #sorted(selected_HUMAN):   
            print(" ")
            print("TF # {сalc} of {all_c}".format(all_c=len(selected_HUMAN), сalc=TF_сalc)) # сколько факторов и на каком мы сейчас
            TF_сalc += 1

            if plot_slim == True:
                with open(outputdir+"/HUMAN_MOUSE_SLIM_roc.txt", 'a') as log_f:
                    log_f.write(TF + "\t")
                with open(outputdir+"/HUMAN_MOUSE_SLIM_pr.txt", 'a') as log_f:
                    log_f.write(TF + "\t")

            print("Dirname is:", TF.split("_")[0]) # creating factor directory
            new_dir_name = outputdir + "/"+ TF.split("_")[0]
            new_dir_name_HUMAN_mono = new_dir_name + "/HUMAN_seq_HUMAN_pwm_mono"
            new_dir_name_MOUSE_mono = new_dir_name + "/MOUSE_seq_HUMAN_pwm_mono"
            new_dir_name_HUMAN_di = new_dir_name + "/HUMAN_seq_HUMAN_pwm_di"
            new_dir_name_MOUSE_di = new_dir_name + "/MOUSE_seq_HUMAN_pwm_di"
            print(" ")

            os.chdir(new_dir_name) # cd

            HUMAN_list_mono = [f for f in os.listdir(new_dir_name_HUMAN_mono) if (f.split("table_")[-1].split("_cut.tab")[0] == 'train')&("_M_"in f)]
            print(HUMAN_list_mono)
            MOUSE_list_mono = [f for f in os.listdir(new_dir_name_MOUSE_mono) if (f.split("table_")[-1].split("_cut.tab")[0] == 'control')&("_M_"in f)]

            if plot_slim == True:
                p = subprocess.Popen(f"{outputdir}/HUMAN_MOUSE_SLIM_roc.txt mv {outputdir}/HUMAN_MOUSE_SLIM_{today_date}_roc.txt", shell=True)
                p.wait()
                p = subprocess.Popen(f"{outputdir}/HUMAN_MOUSE_SLIM_pr.txt mv {outputdir}/HUMAN_MOUSE_SLIM_{today_date}_pr.txt", shell=True)
                p.wait()
            else:
                pass
                #p = subprocess.Popen(f"rm new_log_roc_pr_HUMAN_MOUSE_{mode}_{model_name}_{mode}_{today_date}.txt; rm -rf Bias*; rm out*; rm *SLIM.fasta ; rm out_sure*.csv; rm result*.csv;", shell=True)
                # rm -rf Bias*; rm out* rm -rf Bias*; rm out*; rm *SLIM.fasta ; rm out_sure*.csv; rm result*.csv; rm -rf Bias*; rm out*
                #p.wait() # 

            d1_mono = {x:int(x.split("_")[0]) for x in HUMAN_list_mono}
            d2_mono = {x:int(x.split("_")[0]) for x in MOUSE_list_mono}

            #HUMAN_list_mono = scrambled([x[0] for x in sorted(d1_mono.items(), key=operator.itemgetter(1))])
            #MOUSE_list_mono = scrambled([x[0] for x in sorted(d2_mono.items(), key=operator.itemgetter(1))])

            HUMAN_list_mono = [x[0] for x in sorted(d1_mono.items(), key=operator.itemgetter(1))]
            MOUSE_list_mono = [x[0] for x in sorted(d2_mono.items(), key=operator.itemgetter(1))]

            if verbose == True:
                print(len(HUMAN_list_mono))
                print(len(MOUSE_list_mono))

            if len(MOUSE_list_mono) == 0 or len(HUMAN_list_mono) == 0:
                print("One of the organisms has 0 features!")
                print("Continue ...")
                print(" ")
                continue

            if len(HUMAN_list_mono) > len(MOUSE_list_mono):
                HUMAN_list_mono = HUMAN_list_mono[:len(MOUSE_list_mono)]
            if len(MOUSE_list_mono) > len(HUMAN_list_mono):
                MOUSE_list_mono = MOUSE_list_mono[:len(HUMAN_list_mono)]

            matrix_c = 0
            MOUSE_list_cumm_mono = []
            HUMAN_list_cumm_mono = []

            HUMAN_list_di = [f for f in os.listdir(new_dir_name_HUMAN_di) if f.split("table_")[-1].split("_cut.tab")[0] == 'train']
            MOUSE_list_di = [f for f in os.listdir(new_dir_name_MOUSE_di) if f.split("table_")[-1].split("_cut.tab")[0] == 'control']

            d1_di = {x:int(x.split("_")[0]) for x in HUMAN_list_di}
            d2_di = {x:int(x.split("_")[0]) for x in MOUSE_list_di}

            #HUMAN_list_di = scrambled([x[0] for x in sorted(d1_di.items(), key=operator.itemgetter(1))])
            #MOUSE_list_di = scrambled([x[0] for x in sorted(d2_di.items(), key=operator.itemgetter(1))])

            HUMAN_list_di = [x[0] for x in sorted(d1_di.items(), key=operator.itemgetter(1))]
            MOUSE_list_di = [x[0] for x in sorted(d2_di.items(), key=operator.itemgetter(1))]

            if verbose == True:
                print(len(HUMAN_list_di))
                print(len(MOUSE_list_di))

            if len(MOUSE_list_di) == 0 or len(HUMAN_list_di) == 0:
                print("One of the organisms has 0 features!")
                print("Continue ...")
                print(" ")
                continue
            
            if len(HUMAN_list_di) > len(MOUSE_list_di):
                HUMAN_list_di = HUMAN_list_di[:len(MOUSE_list_di)]
            if len(MOUSE_list_di) > len(HUMAN_list_di):
                MOUSE_list_di = MOUSE_list_di[:len(HUMAN_list_di)]

            from itertools import compress
            a = [x.split("PEAKS")[0].split("_"+TF)[0] for x in HUMAN_list_mono]
            b = [x.split("PEAKS")[0].split("_"+TF)[0] for x in HUMAN_list_di]
            HUMAN_list_mono = list(compress(HUMAN_list_mono, [1 if x in b else 0 for x in a]))
            #a = [x.split("PEAKS")[0].split("_"+TF)[0] for x in HUMAN_list_mono]
            #HUMAN_list_di = list(compress(HUMAN_list_di, [1 if x in a else 0 for x in b]))
            
            a = [x.split("PEAKS")[0].split("_"+TF)[0] for x in MOUSE_list_mono]
            b = [x.split("PEAKS")[0].split("_"+TF)[0] for x in MOUSE_list_di]
            MOUSE_list_mono = list(compress(MOUSE_list_mono, [1 if x in b else 0 for x in a]))
            #a = [x.split("PEAKS")[0].split("_"+TF)[0] for x in MOUSE_list_mono]
            #MOUSE_list_di = list(compress(MOUSE_list_di, [1 if x in a else 0 for x in b]))

            matrix_c = 0
            MOUSE_list_cumm_di = []
            HUMAN_list_cumm_di = []

            
            print(len(MOUSE_list_mono), len(MOUSE_list_di))
            for i in range(max(len(MOUSE_list_mono),len(MOUSE_list_di))):


                pwm_names = []
                if matrix_c == 0:
                    ####
                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_train)
                    if mode == "mono" or mode == "mono_di":
                        for fn in HUMAN_list_mono:
                            pwm_names.append(fn)
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                    if mode == "di" or mode == "mono_di":
                        for fn in HUMAN_list_di:
                            pwm_names.append(fn)
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                    string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                    with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    #print(df)
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_H_train_full = group_selection_GC("result_H_train_full",TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_train_no_NF, "train")

                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_control)
                    if mode == "mono" or mode == "mono_di":
                        for fn in HUMAN_list_mono:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")
                    if mode == "di" or mode == "mono_di":
                        for fn in HUMAN_list_di:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")

                    string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                    with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_H_control_full = group_selection_GC("result_H_control_full",TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_control_no_NF, "test")


                    # split to train and test
                    X_train_H_full, Y_train_H_full = result_H_train_full.iloc[:,4:-1], result_H_train_full["ind"]
                    
                    X_out = X_train_H_full.copy()
                    scaler = StandardScaler()
                    scaler_fit = scaler.fit(X_out)
                    X_train_H_full = pd.DataFrame(scaler_fit.transform(X_train_H_full), columns = X_train_H_full.columns)
    

                    X_test_H_full, Y_test_H_full = result_H_control_full.iloc[:,4:-1], result_H_control_full["ind"]
                    X_test_H_full = pd.DataFrame(scaler_fit.transform(X_test_H_full), columns = X_test_H_full.columns)
    

                    result_H_train_full.to_csv(f"{new_dir_name}/result_H_train_full.csv")
                    result_H_control_full.to_csv(f"{new_dir_name}/result_H_control_full.csv")


                    if verbose == True:
                        print("len(MOUSE_list_mono)", len(MOUSE_list_mono), X_train_H_full.columns)

                    if mode == "mono": 
                        mtrx_mono = X_train_H_full.columns[0]
                        mtrx_di = 0
                    if mode == "di" or mode == "mono_di": 
                        mtrx_mono = X_train_H_full.columns[0]
                        mtrx_di = X_train_H_full.columns[0]
                    if mode == "mono_di":
                        mtrx_mono = X_train_H_full.columns[0]
                        mtrx_di = X_train_H_full.columns[len(MOUSE_list_mono)+1]

                    
                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_MOUSE_train)
                    if mode == "mono" or mode == "mono_di":
                        for fn in MOUSE_list_mono:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn.split("control_cut.tab")[0]+"train_cut.tab")
                    if mode == "di" or mode == "mono_di":
                        for fn in MOUSE_list_di:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn.split("control_cut.tab")[0]+"train_cut.tab")

                    string += f' > out_sure_M_base_1_{model_name}_{mode}.csv'

                    with open(f"code_M_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_M_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_M_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_M_train_full = group_selection_GC("result_M_train_full",TF.split("_")[0] + "_MOUSE", df, pos_class, neg_class, new_dir_name, outfile_name_MOUSE_train_no_NF, "train")

                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_MOUSE_control)
                    if mode == "mono" or mode == "mono_di":
                        for fn in MOUSE_list_mono:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                    if mode == "di" or mode == "mono_di":
                        for fn in MOUSE_list_di:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                    string += f' > out_sure_M_base_1_{model_name}_{mode}.csv'

                    with open(f"code_M_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_M_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_M_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_M_control_full = group_selection_GC("result_M_control_full", TF.split("_")[0] + "_MOUSE", df, pos_class, neg_class, new_dir_name, outfile_name_MOUSE_control_no_NF, "test")

                    X_train_M_full, Y_train_M_full = result_M_train_full.iloc[:,4:-1], result_M_train_full["ind"]
                    X_train_M_full = pd.DataFrame(scaler_fit.transform(X_train_M_full), columns = X_train_M_full.columns)
    

                    X_test_M_full, Y_test_M_full  = result_M_control_full.iloc[:,4:-1], result_M_control_full["ind"]
                    X_test_M_full = pd.DataFrame(scaler_fit.transform(X_test_M_full), columns = X_test_M_full.columns)
    
                    result_M_train_full.to_csv(f"{new_dir_name}/result_M_train_full.csv")
                    result_M_control_full.to_csv(f"{new_dir_name}/result_M_control_full.csv")

                    f_c_tr_h = 0
                    best_pwm_q_list = []
                    best_pwm_number_list = []
                    for k in range(len(X_train_H_full.columns)):
                        f = X_train_H_full.columns[k]
                        fpr_train_H_PWM, tpr_train_H_PWM, thresholds = roc_curve(Y_train_H_full, X_train_H_full[f])
                        best_pwm_q_list.append(score_calc_rocauc.score(X_train_H_full[f], Y_train_H_full))
                        best_pwm_number_list.append(k)
        
                    best_pwm_q_list_1, best_pwm_number_list1 = zip(*sorted(zip(best_pwm_q_list, best_pwm_number_list), reverse=True))
                    best_pwm_ind = best_pwm_number_list1[0]

                if mode == "mono":
                    max_val = len(MOUSE_list_mono)-1

                    # import numpy
                    # list_TFs = numpy.array(HUMAN_list_mono)
                    # indicestosort = numpy.array(best_pwm_number_list1)
                    # inds = indicestosort.argsort()
                    # HUMAN_list_mono = list_TFs[inds]
    
                    # list_TFs = numpy.array(MOUSE_list_mono)
                    # indicestosort = numpy.array(best_pwm_number_list1)
                    # inds = indicestosort.argsort()
                    # MOUSE_list_mono = list_TFs[inds]
                    
                elif mode == "di":
                    max_val = len(HUMAN_list_di)-1

                    # import numpy
                    # list_TFs = numpy.array(HUMAN_list_di)
                    # indicestosort = numpy.array(best_pwm_number_list1)
                    # inds = indicestosort.argsort()
                    # HUMAN_list_di = list_TFs[inds]
    
                    # list_TFs = numpy.array(MOUSE_list_di)
                    # indicestosort = numpy.array(best_pwm_number_list1)
                    # inds = indicestosort.argsort()
                    # MOUSE_list_di = list_TFs[inds]
                    
                else:
                    max_val = max(len(MOUSE_list_mono)-1, len(HUMAN_list_di)-1)


                if len(MOUSE_list_mono) > i:
                    MOUSE_list_cumm_mono.append(MOUSE_list_mono[i])
                    HUMAN_list_cumm_mono.append(HUMAN_list_mono[i])
                if len(MOUSE_list_di) > i:
                    MOUSE_list_cumm_di.append(MOUSE_list_di[i])
                    HUMAN_list_cumm_di.append(HUMAN_list_di[i])
                
                if matrix_c == 0 or matrix_c == max_val: # каждые пять матриц, начиная с 1, рисуем графики  0, 1, 2, 3, 4, 9, 14,

                    print(" ")
                    print("Global_builder is running ...")
                    
                    if verbose == True:
                        print("MOUSE_list_cumm", MOUSE_list_cumm_mono[:5], len(MOUSE_list_cumm_mono))
                        print(" ")
                        print("HUMAN_list_cumm", HUMAN_list_cumm_mono[:5], len(HUMAN_list_cumm_mono))


                    
                    features_c = len(HUMAN_list_cumm_mono)

                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_train)
                    if mode == "mono" or mode == "mono_di":
                        for fn in HUMAN_list_cumm_mono:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                    if mode == "di" or mode == "mono_di":
                        for fn in HUMAN_list_cumm_di:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                    string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                    with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_H_train = group_selection_GC("result_H_train",TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_train_no_NF, "train")

                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_control)
                    if mode == "mono" or mode == "mono_di":
                        for fn in HUMAN_list_cumm_mono:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")
                    if mode == "di" or mode == "mono_di":
                        for fn in HUMAN_list_cumm_di:
                            string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")

                    string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                    with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_H_control = group_selection_GC("result_H_control",TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_control_no_NF, "test")

                    X_train_H, Y_train_H = result_H_train.iloc[:,4:-1], result_H_train["ind"]
                    X_train_H.columns = X_train_H.columns.astype(str)
                    X_out = X_train_H.copy()
                    scaler = StandardScaler()
                    scaler_fit = scaler.fit(X_out)
                    X_train_H = pd.DataFrame(scaler_fit.transform(X_train_H), columns = X_train_H.columns)
    
                    X_test_H, Y_test_H = result_H_control.iloc[:,4:-1], result_H_control["ind"]
                    X_test_H.columns = X_test_H.columns.astype(str)
                    X_test_H = pd.DataFrame(scaler_fit.transform(X_test_H), columns = X_test_H.columns)
    
                    result_H_train.to_csv(f"{new_dir_name}/result_H_train.csv")
                    result_H_control.to_csv(f"{new_dir_name}/result_H_control.csv")

                    MODEL, Y_train_predicted_proba_H = model_building(X_train_H, X_test_H, Y_train_H, Y_test_H, model_name)
                    Y_test_predicted_proba_H = MODEL.predict_proba(X_test_H)[:, 1]

                    #####
                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_MOUSE_train)
                    if mode == "mono" or mode == "mono_di":
                        for fn in MOUSE_list_cumm_mono:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn.split("control_cut.tab")[0]+"train_cut.tab")
                    if mode == "di" or mode == "mono_di":
                        for fn in MOUSE_list_cumm_di:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn.split("control_cut.tab")[0]+"train_cut.tab")

                    string += f' > out_sure_M_base_1_{model_name}_{mode}.csv'

                    with open(f"code_M_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_M_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_M_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_M_train = group_selection_GC("result_M_train",TF.split("_")[0] + "_MOUSE", df, pos_class, neg_class, new_dir_name, outfile_name_MOUSE_train_no_NF, "train")

                    string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_MOUSE_control)
                    if mode == "mono" or mode == "mono_di":
                        for fn in MOUSE_list_cumm_mono:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                    if mode == "di" or mode == "mono_di":
                        for fn in MOUSE_list_cumm_di:
                            string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                    string += f' > out_sure_M_base_1_{model_name}_{mode}.csv'

                    with open(f"code_M_base_1_{model_name}_{mode}.sh", "w") as f:
                        f.write(string)
                    p = subprocess.Popen(f"bash code_M_base_1_{model_name}_{mode}.sh", shell=True)
                    p.wait()

                    df = pd.read_csv(new_dir_name + "/" + f"out_sure_M_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                    df[1] = [x[0] for x in df[1].str.split("$")]

                    result_M_control = group_selection_GC("result_M_control",TF.split("_")[0] + "_MOUSE", df, pos_class, neg_class, new_dir_name, outfile_name_MOUSE_control_no_NF, "test")

                    X_train_M, Y_train_M = result_M_train.iloc[:,4:-1], result_M_train["ind"]
                    X_train_M.columns = X_train_M.columns.astype(str)
                    X_train_M = pd.DataFrame(scaler_fit.transform(X_train_M), columns = X_train_M.columns)
    
                    X_test_M, Y_test_M  = result_M_control.iloc[:,4:-1], result_M_control["ind"]
                    X_test_M.columns = X_test_M.columns.astype(str)
                    X_test_M = pd.DataFrame(scaler_fit.transform(X_test_M), columns = X_test_M.columns)
    
                    result_M_train.to_csv(f"{new_dir_name}/result_M_train.csv")
                    result_M_control.to_csv(f"{new_dir_name}/result_M_control.csv")
                    
                    Y_test_predicted_proba_M = MODEL.predict_proba(X_test_M)[:, 1]
                    
                    if plot_slim == True:
                        print(plot_slim, matrix_c)

                        a = [HUMAN_list_mono[x].split("~")[2].split("_feature")[0][:-2] for x in range(len(HUMAN_list_mono))]
                        
                        if "train_H_1_SLIM.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_H_train[result_H_train[0].isin(list(result_H_train.index.values))]
                            X_tmp = X_tmp[(X_tmp["ind"] == 1)]
                            if verbose == True:
                                print("train_H_1_SLIM", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_HUMAN_SLIM_10000_train_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)

                            names_fasta1 = [x.split("$")[1] for x in names_fasta]
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(names_fasta1[index_f])
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))

                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq), name[1:],description="") for seq,name in zip(sequence_set,sel_names))
                            SeqIO.write(records, f"{new_dir_name}/train_H_1_SLIM.fasta", "fasta")


                        if "train_H_0_SLIM_no_header.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_H_train[result_H_train[0].isin(list(result_H_train.index.values))]
                            X_tmp = X_tmp[(X_tmp["ind"] == 0)]
                            if verbose == True:
                                print("train_H_0_SLIM_no_header", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_HUMAN_SLIM_10000_train_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)
                            
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(h)
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))
                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq), name[1:],description="") for seq,name in zip(sequence_set,sel_names))

                            SeqIO.write(records, f"{new_dir_name}/train_H_0_SLIM_no_header.fasta", "fasta")

                        if "train_H_0_SLIM.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_H_train[result_H_train[0].isin(list(result_H_train.index.values))]
                            X_tmp = X_tmp[(X_tmp["ind"] == 0)]
                            if verbose == True:
                                print("train_H_0_SLIM", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_HUMAN_SLIM_10000_train_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)
                            
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(h)
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))
                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq), "> peak: 0; signal: 0",description="") for seq in sequence_set)
                            
                            SeqIO.write(records, f"{new_dir_name}/train_H_0_SLIM.fasta", "fasta")

                        if "train_H_full_SLIM.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_H_train[result_H_train[0].isin(list(result_H_train.index.values))]
                            if verbose == True:
                                print("test_H_full_SLIM", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_HUMAN_SLIM_10000_train_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)
                            
                            names_fasta1 = [x.split("$")[1] for x in names_fasta]
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(names_fasta1[index_f])
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))
                            
                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq[350:651]), name[1:],description="") for seq,name in zip(sequence_set,sel_names))
                            SeqIO.write(records, f"{new_dir_name}/train_H_full_SLIM.fasta", "fasta")

                        if "test_H_full_SLIM.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_H_control[result_H_control[0].isin(list(result_H_control.index.values))]
                            if verbose == True:
                                print("test_H_full_SLIM", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_HUMAN_SLIM_10000_control_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)
                            
                            names_fasta1 = [x.split("$")[1] for x in names_fasta]
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(names_fasta1[index_f])
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))
                            
                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq[350:651]), name[1:],description="") for seq,name in zip(sequence_set,sel_names))
                            SeqIO.write(records, f"{new_dir_name}/test_H_full_SLIM.fasta", "fasta")

                        if "test_M_full_SLIM.fasta" not in os.listdir(new_dir_name):
                            X_tmp = result_M_control[result_M_control[0].isin(list(result_M_control.index.values))]
                            if verbose == True:
                                print("test_M_1_SLIM_full", X_tmp.shape[0])

                            names_fasta = []
                            seqs_fasta = []
                            with open(f"{basicdir}/all_mfa_file_MOUSE_SLIM_10000_control_no_NF_no_N.fasta", "r") as f:
                                fiter = fasta_iter(f)
                                for seq_rec in fiter:
                                    headerStr, seq = seq_rec
                                    names_fasta.append(headerStr)
                                    seqs_fasta.append(seq)
                            
                            names_fasta1 = [x.split("$")[1] for x in names_fasta]
                            names_fasta = [x.split("$")[0] for x in names_fasta]

                            sel_names = []
                            sel_seqs = []

                            for h in list(X_tmp[1]):
                                index_f = names_fasta.index(h)
                                sel_names.append(names_fasta1[index_f])
                                sel_seqs.append(seqs_fasta[index_f])
                            if verbose == True:
                                print(len(sel_names))
                            
                            sequence_set = sel_seqs
                            records = (SeqRecord(Seq(seq[350:651]), name[1:],description="") for seq,name in zip(sequence_set,sel_names))
                            SeqIO.write(records, f"{new_dir_name}/test_M_full_SLIM.fasta", "fasta")


                    rocauc_plotting(pwm_names, X_train_H_full, X_test_H_full, Y_train_H_full, Y_test_H_full,
                                    X_train_M_full, X_test_M_full, Y_train_M_full, Y_test_M_full,
                                    X_train_H, X_test_H, Y_train_H, Y_test_H,
                                    X_test_M, Y_test_M,
                                    Y_train_predicted_proba_H,
                                    Y_test_predicted_proba_H,
                                    Y_test_predicted_proba_M, 
                                    TF.split("_")[0] + "_HUMAN", 
                                    TF.split("_")[0] + "_MOUSE", 
                                    features_c, 
                                    model_name, 
                                    mtrx_mono, TF.split("_")[0], False, MODEL, mtrx_di)

                if plot_slim == True:
                    if matrix_c == 0: 
                        TF_cut = TF.split("_")[0]
                        print(" ")
                        print("Global_builder is running ...")
                        if verbose == True:
                            print("MOUSE_list_cumm", MOUSE_list_cumm_mono[:5], len(MOUSE_list_cumm_mono))
                            print(" ")
                            print("HUMAN_list_cumm", HUMAN_list_cumm_mono[:5], len(HUMAN_list_cumm_mono))

                        os.chdir(new_dir_name)
                        train_file_name_H = f"{basicdir}/all_mfa_file_HUMAN_10000_train_no_NF_no_N.fasta"
                        test_file_name_H = f"{basicdir}/all_mfa_file_HUMAN_10000_control_no_NF_no_N.fasta"
                        test_file_name_M = f"{basicdir}/all_mfa_file_MOUSE_10000_control_no_NF_no_N.fasta"
                        
                        #ChIPMunk
                        print("ChIPMunk is running ...")
                        if f"train_H_1_ChIPMunk_no_repeats_0.dpwm" not in os.listdir(f"{new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/"):
                            line = "awk \'/^>/ {FILENAME=150;printf(\">%s\\n\",FILENAME);next;} {print}\' " + f"{new_dir_name}/train_pos_id_HUMAN.fasta > {new_dir_name}/train_pos_id_HUMAN_p.fasta"
                            p = subprocess.Popen(line, shell=True)
                            p.wait()
                            
                            line = f"ruby ~/chipmunk/run_dichiphorde8.rb {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats 15:15 filter y 1.0 m:{new_dir_name}/train_pos_id_HUMAN_p.fasta 200 20 1 4 random auto flat > ChIPMunk.txt"
                            p = subprocess.Popen(line, shell=True)
                            p.wait()
                            
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {new_dir_name}/train_H_full_SLIM.fasta {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {new_dir_name}/test_H_full_SLIM.fasta {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/control_H_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {new_dir_name}/test_M_full_SLIM.fasta {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/MOUSE_seq_HUMAN_pwm_mono/control_M_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        features_c = len(HUMAN_list_cumm_mono)
                        
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {train_file_name_H} {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/full_train_H_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {test_file_name_H} {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/full_control_H_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        line = f"java -Xmx2G -cp ~/sarus/releases/sarus-2.0.1.jar ru.autosome.di.SARUS {test_file_name_M} {new_dir_name}/HUMAN_seq_HUMAN_pwm_mono/train_H_1_ChIPMunk_no_repeats_0.dpwm --skipn --show-non-matching --output-scoring-mode --transpose score besthit | grep -v \> > {new_dir_name}/MOUSE_seq_HUMAN_pwm_mono/full_control_M_1_ChIPMunk_no_repeats_0.tab"
                        p = subprocess.Popen(line, shell=True)
                        p.wait()
                        features_c = len(HUMAN_list_cumm_mono)
                        
                        
                        #### SLIM      
                        print("Slim is running ...")
                        os.chdir(root)
                        TF1= TF.split("_")[0]

                        if TF1+"_SlimModel_5" in os.listdir(new_dir_name):
                           SM0 = SlimModel(markov_order=0, threads=45, out_dir_name=TF1+"_SlimModel_0", model_dir=new_dir_name).load(model_name=TF1+"_SlimModel_0", model_dir=new_dir_name)
                           SM1 = SlimModel(markov_order=1, threads=45, out_dir_name=TF1+"_SlimModel_1", model_dir=new_dir_name).load(model_name=TF1+"_SlimModel_1", model_dir=new_dir_name)
                           #SM7 = SlimModel(markov_order=-7, threads=45, out_dir_name=TF1+"_SlimModel_7", model_dir=new_dir_name).load(model_name=TF1+"_SlimModel_7", model_dir=new_dir_name)
                           SM5 = SlimModel(markov_order=-5, threads=45, out_dir_name=TF1+"_SlimModel_5", model_dir=new_dir_name).load(model_name=TF1+"_SlimModel_5", model_dir=new_dir_name)
                        else:
                            SM0 = SlimModel(markov_order=0, threads=45, out_dir_name=TF1+"_SlimModel_0", model_dir=new_dir_name)
                            SM0.fit(input_file=f"{new_dir_name}/train_H_1_SLIM.fasta",background_file=f"{new_dir_name}/train_H_0_SLIM.fasta")
                            SM1 = SlimModel(markov_order=1, threads=45, out_dir_name=TF1+"_SlimModel_1", model_dir=new_dir_name)
                            SM1.fit(input_file=f"{new_dir_name}/train_H_1_SLIM.fasta",background_file=f"{new_dir_name}/train_H_0_SLIM.fasta")
                            #SM7 = SlimModel(markov_order=-7, threads=45, out_dir_name=TF1+"_SlimModel_7", model_dir=new_dir_name)
                            #SM7.fit(input_file=f"{new_dir_name}/train_H_1_SLIM.fasta",background_file=f"{new_dir_name}/train_H_0_SLIM.fasta")
                            SM5 = SlimModel(markov_order=-5, threads=45, out_dir_name=TF1+"_SlimModel_5", model_dir=new_dir_name)
                            SM5.fit(input_file=f"{new_dir_name}/train_H_1_SLIM.fasta",background_file=f"{new_dir_name}/train_H_0_SLIM.fasta")

                        test_tb = SM0.predict(train_file_name_H, models_ids=[1])
                        pred0_tr_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM1.predict(train_file_name_H, models_ids=[1])
                        pred1_tr_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM5.predict(train_file_name_H, models_ids=[1])
                        pred5_tr_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        #test_tb = SM7.predict(train_file_name_H, models_ids=[1])
                        #pred7_tr_H = [x for x in test_tb.loc[:, 'maxscore1']]

                        test_tb = SM0.predict(test_file_name_H, models_ids=[1])
                        pred0_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM1.predict(test_file_name_H, models_ids=[1])
                        pred1_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM5.predict(test_file_name_H, models_ids=[1])
                        pred5_H = [x for x in test_tb.loc[:, 'maxscore1']]
                        #test_tb = SM7.predict(test_file_name_H, models_ids=[1])
                        #pred7_H = [x for x in test_tb.loc[:, 'maxscore1']]

                        test_tb = SM0.predict(test_file_name_M, models_ids=[1])
                        pred0_M = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM1.predict(test_file_name_M, models_ids=[1])
                        pred1_M = [x for x in test_tb.loc[:, 'maxscore1']]
                        test_tb = SM5.predict(test_file_name_M, models_ids=[1])
                        pred5_M = [x for x in test_tb.loc[:, 'maxscore1']]
                        #test_tb = SM7.predict(test_file_name_M, models_ids=[1])
                        #pred7_M = [x for x in test_tb.loc[:, 'maxscore1']]
                        

                        os.chdir(new_dir_name)
                        string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_train)
                        if mode == "mono" or mode == "mono_di":
                            for fn in HUMAN_list_cumm_mono:
                                string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                        if mode == "di" or mode == "mono_di":
                            for fn in HUMAN_list_cumm_di:
                                string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                        string += f' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/full_train_H_1_ChIPMunk_no_repeats_0.tab )'
                        string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                        with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                            f.write(string)
                        p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                        p.wait()

                        df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                        df[1] = [x[0] for x in df[1].str.split("$")]
                        df.iloc[:,4:]

                        lastcol = df.columns[-1]
                        lastcol = str(lastcol)
                        df["pred0"] = pred0_tr_H
                        df["pred1"] = pred1_tr_H
                        df["pred5"] = pred5_tr_H
                        #df["pred7"] = pred7_tr_H
                        
                        result_H_train = group_selection_GC("result_H_train", TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_train_no_NF, "train")
                        result_H_train["coller"] = [x.split("_")[3] for x in result_H_train[1]]
                        result_H_train = result_H_train[result_H_train["coller"] == "macs"]
                        result_H_train = result_H_train.drop(columns=["coller"])
                        
                        if verbose == True:
                            print(result_H_train)

                        string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_HUMAN_control)
                        if mode == "mono" or mode == "mono_di":
                            for fn in HUMAN_list_cumm_mono:
                                string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")
                        if mode == "di" or mode == "mono_di":
                            for fn in HUMAN_list_cumm_di:
                                string += ' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn.split("train_cut.tab")[0]+"control_cut.tab")


                        string += f' <( cut -f 1 ./HUMAN_seq_HUMAN_pwm_mono/full_control_H_1_ChIPMunk_no_repeats_0.tab )'    
                        string += f' > out_sure_H_base_1_{model_name}_{mode}.csv'

                        with open(f"code_H_base_1_{model_name}_{mode}.sh", "w") as f:
                            f.write(string)
                        p = subprocess.Popen(f"bash code_H_base_1_{model_name}_{mode}.sh", shell=True)
                        p.wait()

                        df = pd.read_csv(new_dir_name + "/" + f"out_sure_H_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                        df[1] = [x[0] for x in df[1].str.split("$")]

                        df["pred0"] = pred0_H
                        df["pred1"] = pred1_H
                        df["pred5"] = pred5_H
                        #df["pred7"] = pred7_H
                        
                       
                        result_H_control = group_selection_GC("result_H_test", TF.split("_")[0] + "_HUMAN", df, pos_class, neg_class, new_dir_name, outfile_name_HUMAN_control_no_NF, "test")
                        result_H_control["coller"] = [x.split("_")[3] for x in result_H_control[1]]
                        result_H_control = result_H_control[result_H_control["coller"] == "macs"]
                        result_H_control = result_H_control.drop(columns=["coller"])
                        if verbose == True:
                            print(result_H_control)


                        os.chdir(new_dir_name)
                        X_train_H, Y_train_H = result_H_train.iloc[:,4:].drop(columns=["ind"]), result_H_train["ind"]
                        X_train_H.columns = X_train_H.columns.astype(str)
                        X_out = X_train_H.copy()
                        scaler = StandardScaler()
                        scaler_fit = scaler.fit(X_out)
                        X_train_H = pd.DataFrame(scaler_fit.transform(X_train_H), columns = X_train_H.columns)
    
                        X_test_H, Y_test_H = result_H_control.iloc[:,4:].drop(columns=["ind"]), result_H_control["ind"]
                        X_test_H.columns = X_test_H.columns.astype(str)
                        X_test_H = pd.DataFrame(scaler_fit.transform(X_test_H), columns = X_test_H.columns)





                        MODEL_full, Y_train_predicted_proba_H_full = model_building(X_train_H_full, X_test_H_full, Y_train_H_full, Y_test_H_full, model_name)
                        MODEL, Y_train_predicted_proba_H = model_building(X_train_H.drop(columns=["pred0"]+["pred1", "pred5", lastcol]), X_test_H.drop(columns=["pred0"]+["pred1", "pred5", lastcol]), Y_train_H, Y_test_H, model_name)
                        MODEL_munk, Y_train_predicted_proba_H_munk = model_building(X_train_H.drop(columns=["pred0"]+["pred1", "pred5"]), X_test_H.drop(columns=["pred0"]+["pred1", "pred5"]), Y_train_H, Y_test_H, model_name)
                        MODEL1, Y_train_predicted_proba_H1 = model_building(X_train_H.drop(columns=["pred0"]+["pred5", lastcol]), X_test_H.drop(columns=["pred0"]+["pred5", lastcol]), Y_train_H, Y_test_H, model_name)
                        MODEL5, Y_train_predicted_proba_H5 = model_building(X_train_H.drop(columns=["pred0"]+["pred1", lastcol]), X_test_H.drop(columns=["pred0"]+["pred1", lastcol]), Y_train_H, Y_test_H, model_name)
                        MODEL015, Y_train_predicted_proba_H015 = model_building(X_train_H, X_test_H, Y_train_H, Y_test_H, model_name)

                        Y_train_predicted_proba_H_full = MODEL_full.predict_proba(X_train_H_full)[:, 1]
                        Y_train_predicted_proba_H = MODEL.predict_proba(X_train_H.drop(columns=["pred0"]+["pred1", "pred5", lastcol]))[:, 1]
                        Y_train_predicted_proba_H_munk = MODEL_munk.predict_proba(X_train_H.drop(columns=["pred0"]+["pred1", "pred5"]))[:, 1]
                        Y_train_predicted_proba_H1 = MODEL1.predict_proba(X_train_H.drop(columns=["pred0"]+["pred5", lastcol]))[:, 1]
                        Y_train_predicted_proba_H5 = MODEL5.predict_proba(X_train_H.drop(columns=["pred0"]+["pred1", lastcol]))[:, 1]
                        Y_train_predicted_proba_H015 = MODEL015.predict_proba(X_train_H)[:, 1]

                        Y_test_predicted_proba_H_full = MODEL_full.predict_proba(X_test_H_full)[:, 1]
                        Y_test_predicted_proba_H = MODEL.predict_proba(X_test_H.drop(columns=["pred0"]+["pred1", "pred5", lastcol]))[:, 1]
                        Y_test_predicted_proba_H_munk = MODEL_munk.predict_proba(X_test_H.drop(columns=["pred0"]+["pred1", "pred5"]))[:, 1]
                        Y_test_predicted_proba_H1 = MODEL1.predict_proba(X_test_H.drop(columns=["pred0"]+["pred5", lastcol]))[:, 1]
                        Y_test_predicted_proba_H5 = MODEL5.predict_proba(X_test_H.drop(columns=["pred0"]+["pred1", lastcol]))[:, 1]
                        Y_test_predicted_proba_H015 = MODEL015.predict_proba(X_test_H)[:, 1]

                        os.chdir(new_dir_name)
                        features_c = len(HUMAN_list_cumm_mono)
                        string = "paste <( cut -f 1-4 {out_tab_file} )".format(out_tab_file=out_tab_file_MOUSE_control)
                        if mode == "mono" or mode == "mono_di":
                            for fn in MOUSE_list_cumm_mono:
                                string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/{file_name} )'.format(file_name=fn)
                        if mode == "di" or mode == "mono_di":
                            for fn in MOUSE_list_cumm_di:
                                string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_di/{file_name} )'.format(file_name=fn)

                        string += ' <( cut -f 1 ./MOUSE_seq_HUMAN_pwm_mono/full_control_M_1_ChIPMunk_no_repeats_0.tab )'
                        string += f' > out_sure_M_base_1_{model_name}_{mode}.csv'

                        with open(f"code_M_base_1_{model_name}_{mode}.sh", "w") as f:
                            f.write(string)
                        p = subprocess.Popen(f"bash code_M_base_1_{model_name}_{mode}.sh", shell=True)
                        p.wait()

                        df = pd.read_csv(new_dir_name + "/" + f"out_sure_M_base_1_{model_name}_{mode}.csv", header=None, sep='\t')
                        df[1] = [x[0] for x in df[1].str.split("$")]
                        lastcol = df.columns[-1]
                        lastcol = str(lastcol)
                        df["pred0"] = pred0_M
                        df["pred1"] = pred1_M
                        df["pred5"] = pred5_M
                        #df["pred7"] = pred7_M
                        
                        result_M_control = group_selection_GC("result_M_control",TF.split("_")[0] + "_MOUSE", df, pos_class, neg_class, new_dir_name, outfile_name_MOUSE_control_no_NF, "test")
                        X_test_M, Y_test_M  = result_M_control.iloc[:,4:].drop(columns=["ind"]), result_M_control["ind"]
                        X_test_M.columns = X_test_M.columns.astype(str)
                        X_test_M = pd.DataFrame(scaler_fit.transform(X_test_M), columns = X_test_M.columns)

                        Y_test_predicted_proba_M_full = MODEL_full.predict_proba(X_test_M_full)[:, 1]
                        Y_test_predicted_proba_M = MODEL.predict_proba(X_test_M.drop(columns=["pred0"]+["pred1", "pred5", lastcol]))[:, 1]
                        Y_test_predicted_proba_M_munk = MODEL_munk.predict_proba(X_test_M.drop(columns=["pred0"]+["pred1", "pred5"]))[:, 1]
                        Y_test_predicted_proba_M1 = MODEL1.predict_proba(X_test_M.drop(columns=["pred0"]+["pred5", lastcol]))[:, 1]
                        Y_test_predicted_proba_M5 = MODEL5.predict_proba(X_test_M.drop(columns=["pred0"]+["pred1", lastcol]))[:, 1]
                        Y_test_predicted_proba_M015 = MODEL015.predict_proba(X_test_M)[:, 1]
                        
                        #pred0_tr_H,pred7_tr_H,pred0_H,pred7_H,pred0_M,pred7_M = result_H_train["pred0"],result_H_train["pred7"],result_H_control["pred0"],result_H_control["pred7"],result_M_control["pred0"],result_M_control["pred7"]
                        
                        rocauc_plotting2(pwm_names, X_train_H_full, X_test_H_full, Y_train_H_full, Y_test_H_full,
                                        X_train_M_full, X_test_M_full, Y_train_M_full, Y_test_M_full,
                                        X_train_H, X_test_H, Y_train_H, Y_test_H,
                                        X_train_M, X_test_M, Y_train_M, Y_test_M,
                                        Y_train_predicted_proba_H_full,
                                        Y_train_predicted_proba_H,
                                        Y_train_predicted_proba_H_munk,
                                        Y_train_predicted_proba_H1,
                                        Y_train_predicted_proba_H5,
                                        Y_train_predicted_proba_H015,
                                        Y_test_predicted_proba_H_full,
                                        Y_test_predicted_proba_H,
                                        Y_test_predicted_proba_H_munk,
                                        Y_test_predicted_proba_H1,
                                        Y_test_predicted_proba_H5,
                                        Y_test_predicted_proba_H015,
                                        Y_test_predicted_proba_M_full,
                                        Y_test_predicted_proba_M,
                                        Y_test_predicted_proba_M_munk,
                                        Y_test_predicted_proba_M1,
                                        Y_test_predicted_proba_M5,
                                        Y_test_predicted_proba_M015,
                                        TF.split("_")[0] + "_HUMAN", 
                                        TF.split("_")[0] + "_MOUSE", 
                                        features_c, 
                                        model_name, 
                                        mtrx_mono, TF.split("_")[0], plot_slim, MODEL_full, MODEL, MODEL_munk, MODEL1, MODEL5, MODEL015, mtrx_di)


                        filename = f'finalized_model_{TF}_{model_name}_{mode}_MODEL.sav'
                        joblib.dump(MODEL, filename)
                        filename = f'finalized_model_{TF}_{model_name}_{mode}_MODEL_munk.sav'
                        joblib.dump(MODEL_munk, filename)
                        filename = f'finalized_model_{TF}_{model_name}_{mode}_MODEL1.sav'
                        joblib.dump(MODEL1, filename)
                        filename = f'finalized_model_{TF}_{model_name}_{mode}_MODEL5.sav'
                        joblib.dump(MODEL5, filename)
                        filename = f'finalized_model_{TF}_{model_name}_{mode}_MODEL015.sav'
                        joblib.dump(MODEL015, filename)



                matrix_c += 1

            p = subprocess.Popen("tar -cvzf features_files.tar.gz *.tab", shell=True)
            p.wait() # packing matrices for scaning tool and later use
            os.chdir(outputdir) # cd

        p = subprocess.Popen(f"mv HUMAN_MOUSE_SLIM_pr.txt HUMAN_MOUSE_SLIM_pr_{mode}_{model_name}.txt", shell=True)
        p.wait() # packing matrices for scaning tool and later use
        
        p = subprocess.Popen(f"mv HUMAN_MOUSE_SLIM_roc.txt HUMAN_MOUSE_SLIM_roc_{mode}_{model_name}.txt", shell=True)
        p.wait() # packing matrices for scaning tool and later use
        
print("Done!")